# causal/05 DAG 因果图 实战

数据：5-node 合成 DAG (Z=confounder, X=treatment, M=mediator, Y=outcome, W=collider)
目标：用真实样本验证 d-separation 定理 + backdoor 准则

三步法：
1. 偏相关矩阵：观察哪些变量条件独立 / 不独立
2. d-separation 三种结构验证：链/叉/对撞
3. Backdoor adjustment：naive 回归 vs 控制 {Z} 后回归

In [1]:
import io
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# 5-node synthetic SEM dataset (X = treatment, Y = outcome, Z = confounder,
# M = mediator, W = collider), seed 42. Data embedded below so this notebook
# runs standalone (the articles/ tree is not published to GitHub). Structural
# equations (identical to source repo's load_causal_dag.py):
#   Z ~ N(0,1)
#   X = 0.5 Z + eps_X
#   M = 0.3 X + eps_M
#   Y = 0.4 X - 0.2 M + 0.3 Z + eps_Y
#   W = 0.5 X + 0.5 Y + eps_W
CSV_TEXT = r"""Z,X,M,Y,W
0.30471707975443135,0.09307589385128345,-0.4240282116981235,1.4624738206799623,1.0309793675609726
-1.0399841062404955,-1.2492789894939094,-1.0406614079686092,0.08411763904179148,0.31263772700012127
0.7504511958064572,-0.0392474679326299,0.422235583070318,2.091116726330412,1.2992558857357652
0.9405647163912139,1.1041927359293493,0.5831122360738099,-0.9406087706655695,2.320623649046314
-1.9510351886538364,-0.9725243031269625,-1.6965488385484162,-0.9003706184515072,0.49333999168118337
-1.302179506862318,-0.3108797765628212,1.0294143215948923,-1.1834100109456278,-1.05467265847124
0.12784040316728537,0.7339994437858325,0.12600525778491642,-1.6669318295010662,-0.7844936266646861
-0.3162425923435822,-0.5329627590694007,-1.2708199229365407,1.0064333080361674,0.7064319939687365
-0.016801157504288795,0.7478475848776617,1.4131383898171725,-0.2589255470670149,-0.5935581615203618
-0.85304392757358,-0.04767936032839559,0.6113717634976352,0.34237192988360293,1.3136345913100904
0.8793979748628286,-0.7951140994800643,-1.5030762217919333,2.6061609313548213,1.2671239377809622
0.7777919354289483,1.8312007589007184,2.3265430679279397,0.14726557877577562,1.2456920796967799
0.06603069756121605,-0.4677293949763081,0.5184785498733003,-0.6921312413539736,0.1070233496056755
1.1272412069680329,-1.0914747397011673,-0.3376385375353963,0.0877743434410182,1.2209796852389845
0.4675093422520456,-0.8112890384602796,1.3468320940836955,0.5336181214581723,0.03211989368645157
-0.8592924628832382,-1.4506913921199411,-0.2759096338063417,-1.3495953942618297,-1.5131837640622698
0.36875078408249884,0.2365487106987829,0.042881538105602884,-0.4342931852489884,1.4098473860474756
-0.9588826008289989,-0.7532918059607366,-0.88628765781021,0.6454652991704135,0.9136371625959738
0.8784503013072725,0.1023931830130434,0.6125683942109916,0.5606180085932364,0.010525755419417582
-0.049925910986252896,0.5947390099893525,0.6550246703207856,-1.9528153663691126,-0.6738517295308599
-0.18486236354526056,0.24744417942007352,0.08891339371573334,-1.1996555155078035,-0.2807247521050664
-0.6809295444039414,-0.024416901557684445,-0.7973114243349083,1.4466525247499358,1.0729307562011086
1.2225413386740303,1.0210991266657077,0.43625959216578614,1.0898784169141473,-0.30111107914918644
-0.15452948206880215,0.5388699269678372,0.09186219232821327,-0.6113792433363974,-1.5787392659858401
-0.4283278221631072,-2.3221172528589964,-2.0301094945919,-0.8217330061091329,-2.4593151514865186
-0.3521335504882296,-0.540505027508576,-1.6339638595767585,0.01818736856712266,-1.225897272482069
0.5323091855533487,-1.9140554717127995,-0.00906326270407043,0.1316165798018626,-1.5350434599279712
0.36544406436407834,0.21878195897983457,-2.536253727357739,-0.6538666464860843,0.4171682983368794
0.4127326115959884,0.20173333357364887,0.2659305270333546,-0.689622127168873,-0.4128735531223504
0.43082100300788273,1.2609427762936216,-0.09980022486573803,0.92653921125635,1.2901355987851741
2.1416476008704612,2.2584581931563514,1.9857840648268157,1.0488861640628655,0.42936876856734685
-0.4064150163846156,-0.00043245993994045495,-0.49359794713585786,0.5484812831725205,-0.1876177913046551
-0.5122427290715373,-0.7564824419326686,0.43253948167209544,-1.7722810891885454,-0.6771241038157667
-0.8137727282478777,0.07827474842179899,-0.3734314167112519,0.21899304050792778,0.7435863526416652
0.6159794225754956,-0.2199279260365657,0.04776977567517375,1.1788523206004597,-0.2451116309059609
1.1289722927208916,0.5630934949170969,0.9057645793951803,0.8695947170533536,0.8780064923647788
-0.11394745765487507,0.9291625881333302,-0.18380693988458108,0.5133539256953776,2.920106110396686
-0.840156476962528,-0.9778495844706876,0.7032000091571977,-0.8797103812744438,-0.9463779121282039
-0.8244812156912396,0.39343803139688116,-0.8756664676874188,-0.7496870113826374,-0.8713114361353235
0.6505927878247011,1.002697762604653,0.595402171339775,0.40517791468152803,0.5301754492809302
0.7432541712034423,-0.5831647315291004,0.7476903033291227,1.099878150963974,0.3505338172442698
0.543154268305195,1.2454712623699984,1.0336788705013344,0.33537050473916574,0.6392525028640327
-0.6655097072886943,0.36581149172715793,2.0261883242535577,-1.3483699696948668,-0.5226339396603562
0.23216132306671977,0.21800654890601734,-0.6980373526388552,-0.19578860175519236,-1.4323378233614272
0.11668580914072822,-0.7039805534572882,-1.5903434173857742,0.27233250211238824,-1.1784545007584013
0.21868859672901295,-0.7498617202110636,-1.2192936245584385,-0.3639933813348808,-0.7062627878444697
0.8714287779481898,-0.10194818792892751,0.6298581101697588,0.5799810192420749,0.07374844563413699
0.22359554877468227,0.6543922376793012,0.6942406055731689,0.7128311993385218,0.11742244364891086
0.6789135630718949,-0.6161683858654505,-0.08259385611221938,0.8627086662024803,-0.053743445907922044
0.06757906948889146,0.47130178336431566,-0.9815665140978352,1.8286672092769938,1.21099935173486
0.28911939868998415,-1.0971958295334006,0.4377085444690406,0.5561745032721876,0.6189027933939188
0.6312882258385404,0.11157552848042124,-1.1912321191064141,-0.4619759611829465,-0.2744119515773135
-1.4571558198556664,-0.6189301553538246,0.9732705357707525,-1.7829073618286526,-0.8018155532929854
-0.31967121635730134,2.285294536880266,3.5430267842039447,0.033839616544302634,0.6179667752783633
-0.4703726542927955,-1.6125019826667653,-0.1172994482284791,-1.2138280569482633,-0.8410692962090324
-0.6388778482433419,1.1525756423684625,0.9288808283682819,-1.0547359901602167,0.42465012371647964
-0.27514225122668373,0.012271288373938177,-0.4891783823833779,-0.4214902777147485,0.6612075085094142
1.4949413112343959,1.1586425736331611,-1.462620548996868,0.7299505990039179,1.6577570487630273
-0.8658311156932432,-0.3145676444584171,-0.6970896563677474,-0.9180062009789962,-0.02755129983756055
0.9682783545914808,0.9288659954909735,-0.8142354424514573,0.5090875069091194,0.6205593425422662
-1.6828697716158048,-0.9951199901699201,-1.7455405734659917,0.62465003745882,0.059704250688918076
-0.33488502998577485,1.2866076664645707,2.1083422419350204,0.19174715045233093,1.8026863369099773
0.1627530651050056,-0.3750946116722196,0.1603341345241071,2.338530324013262,0.3296432867677268
0.5862223313592781,1.4253457378588867,0.8072913019716497,0.7550549201766575,0.4617925503209066
0.711226579792855,-0.2887538303165954,-0.39564851429122816,-0.1130696292677878,-0.019480331841829623
0.7933472351999252,0.33642177119867567,-0.07194455108383366,0.39109467191108066,0.7790379118203317
-0.3487250722484376,-1.246325088414438,0.5906531557150021,-0.4647862097397669,-1.679662148341107
-0.46235179266456716,0.22384782401523634,0.034817228509355466,0.37912410327591695,0.2267234002322683
0.8579758812571538,1.8741138975551106,1.0929281072912658,2.865164027799961,2.2756401884487514
-0.1913043248816149,-0.1730083422616615,-0.2479068005235143,-0.9223337392420148,-2.762951156355309
-1.2756863233379219,-0.8347288603728704,-0.5925715488439083,0.26096292679599253,-0.5064769560471987
-1.1332872140034806,-1.681260570069667,-1.911292336622762,-0.4224813272902347,-1.2091887815284172
-0.9194522860016113,-0.6890187965531138,-2.1844547013330624,-1.685650006725734,-1.347627344462443
0.49716074405376404,-1.3442138876946457,-0.6648260673747954,0.5151266637079932,1.001370662583942
0.14242573607056525,-0.8416651186835447,-0.19600284361183068,1.0128858028740226,-1.1301879564136363
0.6904853540677682,0.5720287231430196,-1.114521419765885,-0.19831502434671444,-0.6377657218064601
-0.42725264633653426,1.105386874573129,0.2971411973373058,0.24561684856885946,-1.9366939367483398
0.15853969107671423,2.88848070676786,0.92029677502086,-0.7064594269817173,0.6100939737189308
0.6255903939673367,-0.2737898880872323,-2.2381449724855487,-0.7330163022319186,-0.8990466492855764
-0.3093465397202384,1.2806268867853208,-0.530936593443905,1.5736636891040905,0.9171681116104495
0.45677523755741145,0.4721400991803536,0.5489059234895746,-0.25030907976598454,1.2185928491909446
-0.6619259410666513,-0.4822106172797015,0.24239554421183343,-1.5487441130122819,-0.15485984280902843
-0.3630538465650718,0.2510675284436163,-1.0783546578666758,0.4366421534396898,0.42171156520895725
-0.3817378939983291,-0.12895246331920535,0.11976231150397335,-0.7124672472589988,-0.7913229088289644
-1.1958396455890397,-0.4875240971823013,0.8761499962057817,-1.1998131895663249,0.18252102473255083
0.4869724807855818,-0.1648469224338783,0.7276188822607348,0.6739552244848795,0.23479629118964898
-0.46940234020272387,-1.6328108299002126,-0.5851290359839983,-1.8386525857584493,-2.554001746747789
0.01249411872768743,-1.5373781431582378,-0.9291897562717637,-1.3307302830724683,-3.150302890599905
0.48074665890590895,0.8936182127341814,-0.36125902051425,0.7280030790171639,0.36949615197999
0.4465311760299441,-0.05343450207074135,0.4517894322884347,0.991637456271205,1.2237450942728405
0.6653851089727862,-0.2633963451697735,0.11608506959195597,1.309200282195385,2.7994406580558957
-0.09848548450942361,-0.04073711553155824,0.6171828714427329,-0.83433878621562,-0.3759276046841132
-0.42329831204415375,0.5832830968935506,0.30685874218921716,0.9823289636259787,0.017450515139460276
-0.07971821090639905,0.14050483757319865,0.8609154016194015,-0.9240389020369294,-1.5767322405160793
-1.6873344339580298,-1.4997221536199823,-0.35532411312829454,0.791184977897057,0.19768630844413615
-1.4471124724230873,0.502736719437406,0.9355097319629082,-1.5539120257642738,-2.290203705547702
-1.3226996123544024,0.9178356968044308,1.146173479407968,1.069568509925589,1.544047204769898
-0.9972468276014818,-0.0040667584778875665,-0.8057466577681798,-0.7961375269366875,0.4176094635556248
0.3997742267234366,1.1735506247058085,-1.4560788594590033,2.8938147445857556,2.469962137878646
-0.9054790553600608,0.7892204691337807,1.7802589967546876,0.27931340054704085,0.3146052629018533
-0.3781625540393897,0.9410180393848189,0.5389641425277074,-0.8608509083716528,0.6050242190533976
1.2992282977860654,1.2637133260056563,-1.3353718976944247,0.6400897197344602,1.1228398939983582
-0.35626397106142593,0.4202150615441699,-0.5877549787554601,0.9599390342740739,-0.08804679395146997
0.7375155684670865,0.888418357580046,1.038399799643842,1.4215864712589372,3.0054147367945174
-0.933617680009877,-1.564146514551982,-0.7489207854984876,-0.8266753444485211,-1.547090309110768
-0.20543755786763002,0.5980043779257708,-0.33266010594139583,0.28876078596176485,1.2191348823936772
-0.9500220549105812,-1.8308249985818446,0.8516343607384134,-0.4293104554095295,-3.548682009691402
-0.3390330759005625,-0.9641159930753361,-1.3365219443064114,-0.08825350412820102,0.7502861341791605
0.8403081374573955,1.7237191365367852,-0.28103761679379835,-0.2957172401276833,2.182764692859133
-1.7273204231923487,-0.023440048901976818,0.2043544603776745,-1.3814170632342109,-0.6725947689414556
0.43442364354585733,1.7046101114587913,0.9018379654915196,0.8669234877006349,1.8445378343174714
0.2377356023322779,-0.1528065825211533,0.4505010595843855,1.1355802522720386,1.0499020164830055
-0.5941499556967944,-1.4493238191251647,0.47644089883139745,0.762719730036664,-0.9860856141849745
-1.4460578543884546,-0.9634661242645701,-0.0904612165978542,-0.021582141931343335,0.05459817234138353
0.07212950771386951,0.13809605109750897,0.5308127542676618,0.25800548293599584,-0.610579702855803
-0.5294927090638024,-0.1857576381918915,-0.31493655077265065,0.5786503281411097,-0.41231057867071697
0.23267621135470395,1.2500185844340537,0.4377038785669304,-0.18448132874020912,1.659711771059221
0.02185214552344288,-0.3503443522417241,0.002430874381117193,-0.2872992323740944,0.6558358983922934
1.6017788913209154,1.152923917463382,0.5609753110580507,0.005377450830540642,-0.12722481796883123
-0.23935562747302427,-1.1083049631711182,0.4400483186314253,-1.8518809479443512,-2.2136692990587754
-1.023497492621865,-0.06143146826711948,0.5367162414646952,-0.1054160717501414,-0.7290203724663026
0.17927563495631615,0.09274636766379482,0.07204091319932154,0.26198636149729204,0.9100059310460342
0.21999668397176517,-0.6398149962986349,-1.0010704838306304,0.6979395879472146,-0.9709224849861073
1.3591875752404365,0.44376442544536543,0.20080058569631917,1.0912137565034605,1.1242017300103655
0.8351112459145785,0.23337987644206762,0.04199617803687635,0.5781303600676874,1.152601928889172
0.35687105914950934,-0.09171959976234151,0.16566569988449592,0.5938570035405712,0.1628232515960148
1.4633028912195618,2.502992687877464,2.9574231135036415,0.8060202235886493,2.6645701525979844
-1.188763054322851,-0.692827529469305,-0.7316210373820269,-0.07454432538577138,-0.6879960969173307
-0.6397515327497477,-0.5637718765522116,0.2531574561596961,-1.1942101043641078,-1.092846591452074
-0.9265759414055249,-2.560738928312904,0.3002403887504269,-0.9926105058577742,-2.753920948217196
-0.38980980315576796,-1.0891122535573614,-0.22873176196782555,0.16157619530989953,0.5534093184964743
-1.3766861475563088,-0.9514164982345072,2.052420169701813,-1.8737379055732615,-2.1790976488827547
0.6351509468144043,-0.36827720276944065,-0.4885224525623089,0.19277099244185353,0.26201871255126563
-0.22222269709877338,1.2705962448284103,0.7052680957685702,-0.32308443998134156,0.40243983529759964
-1.4708062945026579,-0.9003409226475476,-1.0689080216872462,-1.1141033495884822,-2.0600051992296775
-1.0155790812075416,0.7806547087144816,-0.11732832252894865,-1.0938755314728348,-0.7657135552024974
0.3135138474501953,0.21846155777546566,0.3144723777292976,1.3843192269659408,2.1842626314114373
0.8381265678943811,0.4562911203687672,0.9831913984183598,1.303001313552197,1.0813037595949626
1.9967308916917865,0.9095689981333923,0.5919655334635918,1.3818876142955117,0.1365779648476102
2.9138624660073296,1.460727484625124,1.6973330947222731,3.0911790666516077,2.5812815629870576
0.4144094332759964,1.9259997196878575,0.4828346053136855,0.6089101002268131,1.1213408783627903
-0.9895381200318641,-2.8143797867671503,-1.1764824658691504,1.844445557568045,2.969079287644629
-2.132046280731309,-3.0675688848043334,1.7837144418380866,-2.4509137418719638,-2.4741174719958763
0.2677114623438358,-0.40931509988419595,0.271443119476772,0.11749212661013436,-0.5265363337033953
-0.812941095310326,-0.3919426657175941,-0.8370643456938258,-0.6310244396815784,0.908544365087518
-0.41535726017968533,0.4824257946741697,-0.549536023845818,0.5505959046452309,-1.324525317633145
-0.6120967990598081,0.16703879849757114,-0.8925682749820691,0.8973150625637443,0.6264970869117552
-0.14079088641638526,-0.4546413560887628,0.8001769490301591,-0.49804559121953645,1.518085021026923
1.0659802307876436,1.552033808635584,0.10123284969743335,2.697080869868661,2.0083580548923687
0.15704856744534462,1.1087196868754032,0.4599783817191066,0.6377560532734041,-0.8600631745736751
-0.1586348370386883,0.10474041275222137,-0.0320865506126023,0.27120841022200165,-0.2079435529234862
-1.0356537528258116,0.44483069607393544,0.8649416167631152,-1.2147498652845097,1.0695584563537497
-1.674682944704357,-0.5646962089200497,-1.016152955560035,-0.10655370131693542,0.040444641202278586
-0.4863079090733309,-0.8046369813195491,-1.3169633805388803,-0.3383817099514085,-1.4103654551479965
-0.05378255081832049,0.6709255635908444,0.4255749762588476,-0.4411192643982148,-0.5703396575827091
1.767929913579883,0.9945780793686192,0.8993327085104827,-0.028130744029047916,-0.07647660348002772
0.13027452147288585,0.06647710749092985,0.3524754453449197,2.666941490125939,1.7279071828624306
0.9827395110230576,1.9643233277011032,-0.2201567307869967,0.8611052803657224,2.750910436972066
-0.49929559853915206,-2.7005199278096628,-0.13267113625150495,-2.6604478442165775,-1.6280608732139494
-1.1849437664170246,-2.010156053368082,-1.3469077431060588,-1.5883854623837008,-1.3432585757667628
-0.9651167622323719,-1.669628807490556,-1.0224272870335502,-0.8044689337354376,-2.56869363067048
-0.7252260645357532,-0.7258736798366823,-1.1617156286730068,0.27109286705568747,-1.5553229203839387
2.1284697324351645,0.8096266964515522,1.5440463575874965,1.2083788485189844,1.5422547936990316
-0.8213866792243861,-1.9180189873852405,0.05801584481177702,-0.348612215907242,-2.4787866995791354
0.838489203736345,-0.5659134304426539,0.4352193465460772,-0.7067227870027031,-0.8618851297912827
-0.9029271780870264,-1.3123052675670834,0.8535198408723252,-0.6260637851258609,-0.6650340801602952
0.9315730128742441,2.923210598513014,3.6896124277926274,-0.1652258064016251,0.24139840069300567
0.38495096610586316,1.9942174308000782,-1.110960874670132,0.515721454071805,0.23070710807707817
-0.1566378976580904,-0.4900683665136945,-0.8853970908312164,-0.9836482722369211,-0.8309793839431264
-0.040762526135434025,-0.38396455337451596,-0.7313644501868786,-1.2079305437513304,-0.2922972610388693
-0.6547876954293904,-1.4765978449029893,-1.2937989758535897,0.5358938374387139,0.591166766681619
0.44607220148208054,-1.6850943124842266,-1.5831115413489805,0.4650179446996173,-0.0029847491217527278
-0.45498348034078,-0.344371926578914,-0.4029646420290398,0.7830144836501383,0.29297787352780663
-1.2256057637672482,-1.6106548651276498,-1.6998001748276999,-0.632670574191398,-1.8678574732165734
-1.2779375743196193,-0.7238193188218194,0.11789657189086736,0.08031938466235966,-1.3926569629242491
0.17258791772211948,-1.5139118901824176,-0.4024752456261106,-2.234582736273654,-2.3851244015571833
1.579091256410435,0.02757116989495967,0.4893000157321432,1.2499833597800407,1.0487128498180673
0.15999161357343825,0.22862290208955,0.5763153222651175,0.20130341045985817,0.551586862902047
-0.11863832610988256,0.30689043472899613,0.6950739838453753,0.5752594794387242,-1.6588270286412623
0.2858261396025429,0.560385028543286,-0.20529342826089653,0.7309838379988697,-0.13999259499642513
1.3060017417068248,-0.6674881133036249,1.9208969686425799,-1.2464292710748053,0.24754885941694083
0.21938250136385634,0.9643769808174899,0.5237442107273447,-1.1512230748003627,-0.014207999187296955
-0.41092723083373717,-1.005675312018719,-2.1641873462304106,-0.8047649909944898,-2.32222906460959
1.1062887100598888,1.186002269637251,-0.6465750648750421,1.5005198151207877,3.699082223977171
0.4287564384616135,0.2037395426126136,-1.6793811262440315,1.2360903890077313,0.6909732334954078
1.535755991995992,-0.6085095030799519,-0.16521822177267825,-1.1175756366605103,-0.4787086105568231
0.18323443722190613,-0.2245804859702349,-0.1891793698444041,-0.9781253567208559,-0.20182816530041092
-1.2244690317205003,-0.24687077379648253,-0.5424869300384025,-1.0808273189679019,-1.0314445125423783
-1.368159199245665,-0.07111421914823346,1.9895918466560443,-1.1941168770310175,-1.3601892299110196
1.6509279322312496,0.6844760686712061,0.7543256946782726,-0.7438471626504495,2.0919894380763187
1.723665720783297,2.3936117509927013,1.4441818867441447,-0.23634372685689198,2.081839881685655
-0.17951921328260065,0.9177511643452586,-0.6557481386385257,-0.3774409669265255,1.0920922998180431
-0.38318732113598775,-0.4482477769677467,-1.1336192244213363,0.018722373981377924,0.3615264902250263
1.4614442922422022,1.4808439582580455,0.4810586865743684,-0.24603520609248553,-0.38591301427890456
-1.107045682043488,1.3802706441664427,0.5870883329021587,-0.013226623498667128,1.9610522020741614
-0.8947270189558264,1.5131210913051139,-1.402375239722923,1.6758581949216977,0.7513569851016566
0.6433267946890444,-0.9063364834779752,-0.4522079678929266,-1.3808746988194014,-0.8603187929980425
-0.3946051228595896,-1.1238356725869454,-0.3230786189025788,-0.7097458405226788,-2.556180662019381
-0.005121866720071296,1.4823578533738153,0.22955819024101562,1.5702516770360742,-0.5145540854425747
-0.16344289852451258,-1.1400025000998935,-1.1553911623766444,-0.6922240274857916,-0.5978220611397278
0.33757454879893356,-1.1537538588922123,-0.8856613395888038,-0.756766601857863,-1.764359497569946
1.4074818613137168,0.2175467999578115,-0.11159515358545258,1.6950781234008039,-1.0328551096179655
0.09058490690174555,0.46551916292545853,0.2304557724876853,-0.1389100962283778,0.7039140362956159
0.6439387932768579,0.21957269002733257,0.6706633895587677,1.2513730612878047,2.8555536467083575
-2.0501721010310225,-1.6756496709211035,0.5781613036745402,-0.9719783682854574,0.06574623897678133
-0.04871840193011795,-0.6985716626895679,-0.5039192223341202,-0.5361049940978753,-0.874656213659394
-0.8432302702928711,-1.1339521995591442,-0.796341855983101,-0.06759105911736846,-0.5919263226413106
-1.218813060423628,-1.4889161588707567,-0.00901917412692077,-0.6389570330039691,-0.17623033915344977
-0.8781523669287508,1.8425566938116864,1.1636119245543601,1.2617374035376514,2.6742750685095755
-0.33412344070081207,0.1304493508351489,-0.3950656280083409,-0.5173897041395343,-0.5370058954778915
0.9159025423560131,1.344710325582745,1.1938065746409958,0.6150996447933224,1.014082700912608
-1.326392717739564,-1.1522738269887056,0.6339031930593614,-1.047123664333861,-0.799083938896747
0.030631492594417446,-0.17064093088884014,1.240494186392021,0.3969823797841698,0.6670470447264967
-0.4841694333335785,-0.9556386980371442,1.491629297308679,-1.851011363120843,-1.5364014725902604
-0.32767309436196085,-2.8155448821001823,-1.7110796032894937,0.13467308774709363,-1.4605211422452886
1.0027578253046041,-0.8766303641955642,-0.24909089992615807,0.4663221540688038,1.174678224691008
0.5381154370039261,-1.5415609614784316,-0.4511905918248693,-1.6818767087581672,-0.40219813267541316
1.3373981074427437,-1.5810847242302355,-0.5723989858015782,1.0915281651815447,0.2326322939117429
-0.15450567924990047,-1.2726118139230531,-1.031046463787052,1.8849059442109501,2.025256135166877
-0.695942611670703,0.9769843735462793,-1.1924666911671877,1.4223382737001462,0.8132155613967957
-0.22385881688049952,-0.15636557614460417,0.04576977318562255,1.2881269370673532,2.188854129090755
0.2424967912712216,1.4118121579567848,-0.3448120121608771,0.791807684598844,1.6443253648133647
0.17657335845371103,0.4992692561277308,0.23577425758594425,-1.4559800612684932,2.7926707196442084
-1.0843880722333665,0.24036478959847596,-0.7219341048193106,-1.9289041360342822,-2.0367516000003807
0.09048978162787422,-0.855652104490638,-1.3563539771915327,0.21123893199817206,-1.0115518384635744
0.22822833013890514,0.6380651192621234,0.4257178961555347,-0.3534278402145661,0.5611164677243263
2.5174740375339204,1.9874512528413062,0.7471569026588878,0.4862150535375699,1.63406720616843
1.8768446112816701,0.36177501340714213,0.6878144753430756,0.5074538097100675,2.3847533957336235
-0.8532433505588201,0.13291869533675632,2.225739375018652,-0.032368114175385765,-0.6180993927188486
-0.2873833615491761,0.4224960778002449,0.4040383545509203,-1.4507817950169861,0.3907756600855137
-1.4634420018370031,-1.2812736220485843,0.1006996804495543,-1.4205596057408303,-0.1884536092127116
-0.5907070139634865,-1.4181219392527382,-1.5029979106431468,1.0660093853588033,0.5285915552736801
0.3156050035903405,-1.026325830103102,-1.594805189567305,-1.0721761732720523,-2.0102473436118973
1.2058536208882336,0.6894323523655785,1.7896591524830072,1.1493583838810564,2.3271541733775156
-0.7290838377436085,-0.03753205008248944,-0.9347393949580505,-0.5315208539134622,0.944798479317438
-0.6541464400677965,-1.1195990504918127,-1.379823786616186,0.17767496495612584,0.12862247702253393
-2.147289029738655,-1.0470591735432289,-0.35321667327952166,0.10496046351988353,-0.26670166780271853
-0.16266592054490767,0.48920407586603154,0.056568809382421106,0.8768498592914776,0.9091349776231259
-1.062414411859563,0.0869835087355193,-0.8764114538226359,0.8005109729147425,0.3650114485857532
-0.5294394273660737,1.2871604064394517,0.0811749032187355,-0.8117961796857447,0.8784672476294954
-0.8768607781675882,0.8385504431142492,1.480940850584379,0.8898334804198462,0.8953268147120252
-0.09426255425255699,-1.0484227098828225,-1.1205204217966442,-1.0762368805043872,-1.910854770546246
-1.7577283913566313,0.8047220138597069,-1.1522558802082596,0.28456322975008386,-0.5039490230226966
-1.4670452453909906,-1.26449912725121,0.5632438827244293,-0.8003158140654402,-1.7844437243106173
2.129247112028298,2.1094963684685766,1.1251458224786095,1.7426660753697791,2.1482134644686295
-1.287422581274031,-0.5757246645859266,-0.7932360200244396,-0.12091276015150143,-0.37527454880392136
-1.0967855784546396,-0.9605421676842874,-1.870329666432204,-1.182996585848389,-0.8564255834775891
1.836913528321314,-0.8892400956142757,-1.0113953851228459,-0.952210315020463,-0.41991207177125833
2.905067169240407,1.2809690744038553,0.3295862854718853,2.190339203401056,1.1083332129855767
-1.1715666288253417,-2.14516654971828,0.5221221249683351,-1.128126456009666,-0.1709863345186251
-0.36824895678803055,0.7831747574487571,-0.8016295476350299,1.5124979664279175,0.880125534890863
0.34155555094050954,1.687570432609503,0.4217512482713774,0.6484116544838107,0.7345640587651477
1.7286976444055924,0.06743783290610594,0.6953760097667299,-0.9986551361461863,0.016777585712412224
-0.9868570784282374,-0.19145779669484758,-2.438704679130375,1.0812303746237708,-1.3916239134106112
-0.24527784594210975,-0.8480304449790302,0.7793770344451907,-0.9390415483243902,-1.6840704463746294
0.777337576061744,-0.2398159421059176,-1.2543953548382127,0.7301574243648199,1.226243539821284
0.43476607446661863,0.9918777410986807,0.6295588157865388,-0.24619027285736916,-1.3645781949414548
-0.37615607123009925,-0.22661629745846212,0.1786982603732054,1.4050687178066348,1.7068004071072422
-0.13382296451604114,1.672736559672991,1.5864073593657424,-0.5648985615419613,2.0650555654246494
-1.3748958083699818,-1.1696803828879119,0.20002026602716677,-1.2577747029777673,-1.583182880756588
-0.23817374397466523,0.8802494604871566,-2.113215653583834,0.8107394944165462,1.1373072448781614
-0.2663874900089551,-0.035780332046695576,-0.79551334677,-1.0273080375543229,0.5875429000505173
0.23216988962625595,0.9118629545573611,0.4117640157204884,0.6825628741543187,0.33042372631232025
-0.555327218819016,-0.7248212667394547,-0.41242626614162337,1.2378690583543281,1.1242994163336182
0.471538522545139,0.18621042372098678,0.5334738568730665,-0.38121120581638734,-0.7686491861783719
1.0127158178198286,0.3908590296109795,0.9066242645529992,0.673976323311466,0.054815383128043405
0.15542932766846604,-0.7583424510115542,0.3809490011262129,-3.08233336836788,-2.7196743740146907
0.35175640839920347,0.8379597387813544,0.0371129790799411,0.39435902173147397,0.9588065794512177
0.053155347577867364,0.6768266793672979,-1.643289007310291,0.30229279155613226,1.6475287880272647
8.439309141418043e-05,0.5935968914512738,-0.2774021814194352,-0.5255530916185641,-0.15009103220080944
-0.7215580335428474,1.1761866371462208,0.7692241392828214,1.1816128865676019,-0.09234939805549192
0.316494261673308,1.6075919320729575,-0.6081696729285904,0.5333753042930653,-0.012223204103804397
-0.09728659841348947,-0.42358798905588013,0.5041983334048122,-0.47693832892094884,-0.05566373761787413
2.093168308930595,1.4196364728994757,1.3342842391371472,0.5517553592224681,2.8298123294552413
1.5733549024752425,0.16307676197940857,0.17579000488966512,-1.4147366972086735,-0.23210496830023486
0.3858465525565008,0.21627859239280586,-2.9988962885945543,-0.32994177963397764,-0.44372814427183854
-0.7630572096947675,-0.9812780539662485,-1.572471778901678,0.6541594574056222,0.602624668569971
-1.112411471983418,1.0638645731443421,2.2053485304993843,-2.459197965317137,-0.9371653316731438
1.191142953088865,0.9592696268873954,-0.2587569049621594,0.026486768058344046,0.11459298010866331
0.2627492251471385,-0.08665451553085288,-2.0101395777882516,0.04647082059968688,-0.9785509531608466
0.4801434033916108,1.4336902735521824,1.665354470375927,2.128070103654736,3.189329032849815
-1.7445859869741926,-3.042908403282508,-0.8995169294410795,-0.7920320060206761,-4.456678122147256
0.9274384815581808,-1.5503833540861993,0.6398254827318657,-1.0270574207664982,-0.4908611871470796
0.45442033821436295,1.0230488243269553,0.5838733225174519,1.7332846638071517,0.9144218062695166
-1.110430684414478,-0.49474689408987094,0.3101350679852075,-1.333570797058453,-1.3244995770257746
-0.47152480744994896,0.014683896535786167,0.36741441667525243,0.8000330367782849,2.099673622774848
0.2637172043565196,-1.194537501140289,0.3391270615789133,-1.8254024973879581,-2.224101091464175
0.05246679835624363,0.002999688327755611,2.278330355054425,0.6345316756162476,-0.36130347205640856
-0.292171185803555,1.8574238174995645,1.5282334812090574,-0.056462170454037786,0.770086584603233
-0.10348826806596086,0.7724554857628825,-1.3486026721166702,-0.48084799263257094,0.5596511323773941
-0.25197737820688537,0.043717856039369424,0.4597768720907142,-0.3672050585407748,-0.6035993835435721
0.15256251210246857,-0.3627726540537034,1.0326489718871068,-0.028525922647947344,-1.7695082188867108
1.471491972993157,0.3411731346271847,1.3050527469243014,-0.1778421341617515,-0.7840326806490489
-2.5666584409312976,-3.4125271096656666,-1.4503631159587993,-1.2909822469585723,-1.0704208218476001
-0.23685026450968424,0.13921528866631844,1.2067090317880231,-0.8397046539080499,-0.2754678934853543
0.17651242137244696,0.9337393191680774,1.2078641953016285,0.8052490089499973,1.3205814477628595
0.2959939896870995,-1.4276223323270654,2.1442307372716103,-0.4688748214424189,-1.2096697840577797
-0.37191458132128985,-0.8473741829674609,-1.0042139911387498,-0.3106043040635564,-1.2384869396887783
-1.7567217824785826,-2.035825523472713,-1.118148233442239,-1.559099409810162,-0.1198930167660266
0.32799548371410964,-0.8013534529517087,-0.5465250843454665,0.0387030765679089,-0.5695170452429805
1.727350214164185,0.9171567339039928,-0.21507512986757749,1.4783365268180675,0.7543767508339032
-1.5338614049161376,-2.8513328574465433,-1.745420606997566,0.810191709677663,0.0755717924708994
0.8638280136981883,1.0461844452061582,-2.448820172111896,2.10680663841248,0.8541351829419339
-0.3285252231228939,0.5895924583722353,-0.25782173621146354,1.16483371347016,0.282485742147201
-0.0613243458288473,-0.2817647609513361,0.2848286314145036,2.9004432604481942,1.319585933765327
-1.052898510703949,-3.0071577653365162,-1.3953691751068655,-0.7237003475565743,-1.8651589523583656
-0.33445617235587666,-1.163646712180328,1.5862871698750114,-0.3984094592421941,-0.5214167378080466
1.3000445946989585,1.8829242314521757,0.5782530821515153,1.7315475288819089,3.89890496956253
0.582655241987707,-2.486666659734719,0.5450290572637199,-1.9378418803385726,-1.8760060236842753
1.732311605409944,0.5186323905114528,0.4170101298834042,0.1998427754321273,-2.01881637977901
1.1774118889776937,-0.5912948339784122,0.6490188596997343,-0.5192674055303377,-0.2289821763228141
0.4390866789295762,1.0241136410496248,0.5864756462484497,1.0716903080685944,1.0857819690487085
1.743934526167792,0.19685331094043468,1.4278406251856932,0.14217899496817413,0.21296617730576206
0.43899315873829875,0.6234506001047609,0.242974530163829,0.470706707104931,0.7930259630010167
0.8279881767970589,0.9794545075676225,0.7899570603590724,1.5696414641428167,1.2458696981804203
-0.29657095353470836,1.6879403629626657,0.8727150425388434,1.2748125893210238,1.4287643830864556
0.06654581585920483,-0.1696690357457678,-0.33918616885104913,-0.14178763881158288,0.45731587249211075
-0.6974238233002769,0.027332457972472912,0.4131820563164187,-0.9509256719426,-1.0038774041795646
0.989583934594853,-0.9902635118300094,1.9373071926486791,0.6485830687699048,-1.1185265571692278
-1.1783036231148656,0.6012617517363112,-0.09001841657454865,0.6789084779848402,3.663866567482967
0.7823503424176613,-0.369333676862534,0.3086647897944881,-1.5810181442739952,-1.691033354349498
-0.19065105750515945,-0.6560796928403253,1.0609550430173451,-0.8449721418994278,-0.6503830049098002
1.1712470934673194,0.563375751884449,1.3092660065096149,0.4803866895088148,-0.9737790301787355
0.7508689887312598,-1.1874498509742195,-0.3445185375729006,0.5980141808653016,-0.17824942000303706
1.8206461576468016,1.1390931077523387,-2.04166077063166,2.320264739461079,0.7964533729173736
0.7307746911386448,1.3328536184943331,0.9165589041967661,0.8529680750754469,0.6504379039348791
-1.5720402556532407,-0.49308630432932876,-2.3674001499218513,0.3898809629155226,0.9934619950642207
-0.06695317289820084,-1.6223741506278255,0.9832249857528834,-0.8015438068930135,-0.7399494103823457
-1.172007192594816,-0.7098128906270675,-0.04346999599485968,-1.3006850030108605,-0.43683942224414496
-0.5182798423662747,0.4828889817963533,1.144540714410895,-0.45250089391477405,-1.7560107944510732
1.5112284332767216,-1.122374549327547,-1.2388235029793153,0.271174597487969,1.4112951939092238
0.637533810897696,-0.755999108368866,-1.779307205810714,0.21004269106531528,-1.4648595195936505
-0.6989304285787463,0.5270783897921468,0.41887790107881123,-1.968196257315998,0.8080018805319393
-1.013717366440399,-0.2367632490229129,-0.10463241466301584,-0.7753773615381129,0.5154735607009733
0.03278209287626079,0.4283626285648842,-0.3630320375877004,0.3099917398955239,0.9465099260148842
-1.2165601493811278,1.2512147329327477,0.4600257006125881,-0.2378572836694578,-0.3483060562767366
-0.6711402773965811,0.13298263980791203,2.8073306839746337,-0.956438959602322,-0.26013831507768204
0.31200947155124387,0.8138927462088013,1.0786933712316982,0.6914126365071415,0.2817312807131116
1.1553120361066962,3.2884015452716744,2.036019069444821,0.17579322800308073,1.7611369581426695
0.6087614722244398,0.09057617802172918,0.8711097449598744,-1.5834052500671625,-0.9962867951082766
-2.2912895085981,-1.943527517917545,0.8964038884769294,-1.9922484282785087,-4.0064984796704355
0.3043667290302598,0.5799797627622626,-0.016077744072066108,1.0906318781301858,1.9570899714308123
0.07203357323257792,0.4750078160051838,-1.9122589628737385,2.281618030433233,1.2972057666698473
0.41389028537269307,-0.31977024925986164,-0.2845172077839683,0.835290666559397,-0.22704019085246935
1.6162096816512073,-0.2242921787292439,0.7444480563777286,-1.2220073381407284,-1.283476991264754
-2.063238276804743,1.2782252836877066,-0.5863328100087617,-0.22278204694135895,0.8861811947399421
-0.5911034251911056,0.03129175779247012,-0.1669784231515893,0.12521574127150364,-0.8155025887624854
0.5909063227933546,-0.43683222000539834,-0.8228194090039326,0.3446940075602257,-2.30124466469663
-1.5815943994539423,0.12513310255262744,1.5395691826089186,-0.19303436239404137,0.37317917328756917
1.475949048053509,0.6288293724654199,0.9381058271461897,-0.4103536484877781,0.6283408599456426
0.3683566127006584,-0.06870313996750207,0.6309296786537084,0.02178287964319761,-0.22309195483978514
0.846583985843404,0.9855648648361952,0.9172707725718299,0.8850698169492623,-0.2125855342972801
-0.5709436615217945,-1.5773540989925872,-0.5104554822499031,0.12553731170006688,-0.6369728527569748
0.8137636899662105,0.7595090489002776,0.4254089971975868,-0.3428672395023352,-1.2562114636146176
1.0684715559890383,2.4471905857262715,-0.45102740481443615,1.0444663148072093,0.7185978535790809
0.23287802013716835,0.582616672265895,-2.0116569770707993,-1.998576209711672,-1.4586777052270496
0.23440089269869593,-0.2590116574341165,0.07237763311478912,-0.6692218893647699,0.3798222974921128
0.2703432392817259,-0.37193366078737905,0.21241015803593066,-0.634567417108663,-0.8658614893849381
-0.8633452647393817,-0.8420126057148002,0.9566607867109376,-1.8067252872173651,-1.7722353582425432
-0.14752868024035293,-0.3277270181679575,1.4730177106319804,-0.30434080323303325,1.6386586047107508
-0.1525225324523588,-0.4540677989844514,1.4337781453036114,-0.5700849404354787,-0.15961387252125114
0.3833938643534951,1.1080083732513142,-0.6656210813184835,-1.5678681527073404,0.10906462074681295
0.9998242469102946,-1.2332969797538358,-0.72225412887592,0.16658727724849454,-0.9030555964614455
-1.0585360819784093,0.33273200942292935,0.5801538987082584,-1.22882293135503,-1.044417128602828
-0.12500903031957764,-0.44219104030234996,2.3992473656514726,-0.20853963915659268,-2.511771779746622
1.4814555476589732,1.1176755595248589,1.8154018715713738,-0.7534247339353046,1.1508659447183354
-0.7435882288125641,-1.498782024607146,0.6697625546162365,-0.9265352508789347,-0.7643489319866612
-0.8222500172903203,0.917506058204276,-0.11253613808800778,-0.14032868317429006,0.20942269430157964
0.20230619183119358,1.3467880560874699,-0.4791632143288836,0.9983416078004532,1.0353381301077058
0.8443851904853201,1.3817831209927336,1.2034428196810596,0.6900375192578243,1.772054582386334
0.01142605556065933,-1.5090417480132097,-1.553373788638323,-0.5446782998308759,-2.5815700970018955
1.3289605904369892,1.4936013662845475,0.2096440589050616,-0.5820724021803412,-0.13411684950170233
0.8567939825665268,0.8332520789189723,0.13519767467376909,-0.9139084720582384,-1.1766310594621636
0.8418200668900618,-1.1843612136875368,1.2376037250165468,-1.0526061495035246,0.6530256008912156
0.5541165013240933,0.2527608668912312,-0.2788690428923082,0.31483322180259515,1.159452576200214
2.327653100346085,1.528361039206305,-1.980741326810724,1.0975406639493754,-0.8897417895575241
-0.2051616888660442,0.45349442939012874,0.640492976479507,0.6169626003798024,-0.97853311175501
-2.003522292066514,-0.8244997790844012,-0.9463717868292467,-1.020803161099549,-1.0549971434494614
1.604254361517141,1.0933585976059501,0.37063860371953183,-0.12052069802589438,0.013978737717603162
-0.45769943095682564,1.2447616767746827,0.28672380219028115,0.0251537879968598,-0.6894244756911171
0.10788044434303304,1.2799741320821965,0.11028824380689067,0.7481559736909628,2.958827290537069
1.309550684860527,-2.21227961135248,0.12665669837896,-0.1924932599340416,-0.8898465216606816
-1.6022595410589253,-1.1185683253671446,-1.8684353759628505,1.323961014721164,0.5775126675401012
-1.2516472141061394,-0.790474634597021,-0.7541566153334666,-1.2097620350073763,-1.6921332840344125
-1.6012779371188537,-2.552737541286398,-0.08178334926853814,-1.558600389386908,-1.3333948181586508
-0.7941362900254239,-0.30288508678039977,-0.7049503013784881,0.20771908537533698,-0.4280157616845432
0.43963660601699356,1.4685760754190034,-1.1682338866442707,0.9295452301007383,0.9157174790523714
0.524187842812771,-0.8244739381842048,0.4212131618014744,-0.7573940007787987,-0.1799320096562419
0.27627417932855375,0.47453802835414916,1.2654560389477838,-1.5296509189935745,-2.1565540103800123
-1.4127658838923665,-1.6222064192426209,-0.5316571334157926,-1.4772154652346154,0.04731691761236445
-2.310103436632644,-1.826963741995999,-2.508934721068332,-1.9957319962923274,-1.7289586544202875
0.054353585390382264,1.5045704268652262,1.157340874758384,-1.33994060971225,1.3632591173569124
-0.47177603363833576,-0.7784458369309928,-1.8934216324989859,-0.8699056600346871,-1.8995955051016584
0.45938577171661177,0.6919053748689306,0.5163708893088449,-0.7357880171656677,-0.9451064451765616
0.7019536262212701,-1.520241302950352,-1.3435362156272654,0.4549833551046414,-1.6945291955226676
0.13824143205868192,1.9314382774242211,-0.5804078025396989,1.5312211385095265,1.9816273806317126
0.7601330853758538,0.9827189889944259,-0.14709432038136244,-0.25333290429407584,0.4306763442684211
0.22921137411506842,-0.06789415559272746,-0.2036658031050781,-0.17788971994467595,-0.5873037569998822
0.5300647056014667,0.8405788628006281,-0.3920345915462598,-1.2845847870025922,-1.2360721549862614
-0.7046732628698824,-1.7621560603827515,0.06005646010032528,-1.4226156452168244,-2.85898212253063
-0.17961141383145854,1.092725034836928,-0.11825186800775539,0.8058388640892303,0.9007980489924231
0.19677609665695725,-0.22559672716795154,0.9125118535214365,0.611229657039722,-0.2727703146527149
0.8205284754174188,0.2061947226602747,0.34303158244156184,-0.45301696649029066,-0.8090181142732703
-0.3937411722245721,-0.6885262010777705,-0.9041150669196392,0.8391450418129678,1.6223989064992081
0.5211672557321126,-0.32006777487952837,-0.9619602162961445,-1.2949768114864204,-0.8898479713757084
-0.265838791915379,0.5617512156707191,-1.478970934346223,-0.806884763580874,1.1474761599718477
-0.11754216732813036,0.19713576644015768,0.34987390779886973,0.6832787404557262,-0.0930449292347546
0.829519042024073,1.0665076624150764,-0.034635248526835194,2.3535070558418667,1.9547836429271284
-1.993060371054156,-1.0053248750126493,-2.6737929905673092,0.028142524975850736,-0.5440671912747342
-1.296472328074761,-0.13503635296019945,-0.5762257786574599,-0.36805347040902847,-1.9491672212027367
-1.4821853974428207,-1.1599414426464487,-1.2490164620488866,-2.854012650403857,-0.5786387402612398
-2.3336161198483047,1.062923781485396,1.894554726616279,-2.676273324727068,-0.9418283786532519
-0.6782644401551767,-1.907062636508087,-2.2748974028457356,-0.38191536918477165,-0.3980707832134689
0.7494338997277404,1.6356542734268547,-0.9506545177059974,0.711160113259089,0.42137847088141256
-0.28488406638257474,1.2254654175296833,1.4776092683646047,-0.996636088727938,-0.05042971717889197
0.19779008194185213,-0.7532637649946677,1.1307840279655628,-0.40259270617837334,-1.4447805955098754
1.0892174967107926,2.7738823589113695,0.0871817689477693,1.5340132062960414,3.389946265367099
1.3276861322676916,1.659333349079164,-0.1342858180851617,2.033312211469278,2.1577321277248966
-0.06913793472613955,0.9365700394598417,0.8972064083912654,0.31104763670054253,0.9908623647610804
1.3535858895693973,0.7048151318105753,-2.1156552574021377,2.339527239409861,2.8803553327151974
0.09212665843410921,-0.17116282900162183,-1.9125425332060426,0.3496754021196059,1.245597583458015
-0.8373982238274621,-0.6281079389533392,-0.4691657825367136,-0.5239644810295677,0.1893063261448319
-0.5944003521987352,0.09333042022014959,1.5778896921951708,0.6077463102365873,-1.0711889818090943
-1.4805365125650163,0.660918018751388,-0.23745468254132318,-1.1107436898021195,-0.9205919550420494
-0.8881338537336524,-0.2627527090054066,0.022110979033532638,2.4384840177371965,0.08397751835454215
-0.35801668807442916,-1.090451967174505,0.8066536824116605,-0.329659051607727,0.01747818423468317
0.8035850193786016,1.2262195720742806,2.4747506049004304,0.6174424665008797,1.0924829610376317
1.7207698311659838,1.781535661977784,-0.04238777455497733,1.7624187430245017,0.38146476887054814
-1.38218151537704,-0.03643122924073394,-0.02776609250564624,1.4534627475233661,0.5197715461300885
0.39282746825991893,0.9419126406549421,2.1046290135393253,0.8661949057046942,-0.5900585625389603
-1.0405439391575344,-0.7912000997365753,-1.759559305112401,-1.8817873396233338,-1.3896502077497146
0.47469708846320197,-0.6916406228989785,0.8200888836841282,1.105205485410048,-1.44859007273165
-0.1310866506877266,-0.9337367559846431,-1.2984365591566511,-0.4940299652138623,-0.18505586721463985
-1.8309058258475304,-2.550916161381623,-0.6628024813113301,-1.329922643205319,-1.5624567762593975
0.9282969915684843,0.7010971073301919,-0.032289915201216685,-0.7138680487324011,1.901563022645777
-0.605000712808731,-0.3859491396309591,-1.073796342590517,0.4050814169469961,0.8574394281553609
-0.5339002383735546,-0.7069378237164861,-1.195183274773221,-0.8552828361115689,-1.2040245857365302
-1.06975241128969,-3.4904950299899147,-0.7707341447432199,-1.4628072684407551,-1.6585810559408487
-0.6542832766875555,-1.5744591080379466,0.5232015957512981,-1.2148079507324878,-2.792297410330396
0.4278904406949172,1.3347858717506254,0.2706995583366844,2.53315437097724,0.9714975176828472
-0.18924434093640552,-0.7592481705313338,-0.15766129242618376,1.6533199881373712,0.7801784408781287
0.32866200228248105,0.524922678798082,0.7286508699304718,0.20935066843262412,-1.791112989132574
0.3619218539288437,-1.2097245030108343,-1.3345361239162101,-0.04442830887358391,-0.6642371908760922
1.320661655528167,2.0990363101233145,1.3694034014499175,0.5884893740682773,0.3234995499773752
-0.3427861508643793,-0.31542073548686606,0.7247168764136871,0.7320587767659885,0.30962607579282475
-1.4768578168457318,-0.4483885119393307,2.469109745168611,-1.445154569196551,-2.0245078986152043
1.067222416571983,2.8593910001676766,0.4817131094907207,0.9462432284948219,2.327065251786438
-0.3314881720972547,1.3506960145407763,-0.05287299730589473,1.4299994173132773,1.4096751395944165
1.114592444577377,0.2507306010016307,0.8441850518742348,-0.272466131917631,1.3894981514233304
0.3833771131824704,-0.40153162827912403,0.6986446804284011,-0.22308348789111931,-0.30557302398349956
-0.13113753020334493,-0.3911137766212043,-0.6524212863630733,-1.6466132939478633,-1.7875063402459999
0.3487758940462951,0.6842861954996832,-0.19787681828014392,1.1221129551929576,0.6563137195190376
1.9510125601262995,0.8702821458122787,0.802215481961724,2.7138404691523252,0.4478249104655705
2.076980529053753,0.6432461195234049,-0.63461040487764,1.7285400603008079,1.5920303446175463
0.0693811351327787,1.5018718182754316,0.04002755746761455,2.0753567239037887,2.3784645174048826
0.1601905933170753,-0.034114595119299235,0.07935838883264487,-0.827562473996457,2.03835750213494
1.0762401574663856,2.837709171770512,1.8472514363534116,1.3248297107056388,1.4594334506336706
-0.8456610327673472,-0.3320879526315006,-0.704225001968991,-1.1888484148583958,-1.0282627574201308
0.3330703726182551,1.951853580533855,-1.2568283878705153,0.9042626759917788,0.8522996038246012
-0.0258628479538564,0.3443879987336986,-1.8132810678981013,1.2648640360394476,1.016739718517797
0.3139082114575536,1.1199146789782246,-0.9234386630126661,0.8286466807343877,1.35597121433592
-0.8333688059494058,-1.1412954544924272,-1.6555563545476655,-1.7880585752734812,-1.518051859311465
-1.589567493308969,-1.6485429692418962,-0.6491736493924779,-2.321481811348707,-0.8986864734222695
-2.0729834359912918,-0.5767857629806583,-1.3619158824918025,-1.961390090982064,-1.886067430736139
-1.1173841129896078,0.42986118597051126,-0.5737966040300507,0.44294115123692257,1.422575216494429
-0.458675284939333,-0.18908114046920366,-0.8308102271228108,2.0168215394117492,1.4523680531123553
-0.2931915866487618,-1.432989128408078,-0.8704045993704168,0.8753860365164364,-0.2908314658447673
1.9372311624295169,-0.3916427850745686,1.2050114960007519,-1.7124752636471872,-1.3669956651180233
1.1059933699072981,-0.8741397172325921,0.39970567426073905,1.3232769272967886,2.6512363637634677
-0.9620911163416273,-2.2486676711101947,-0.6584708860570545,-0.11999980912050812,-2.0013423908574306
0.34770845245095694,0.283339585672566,-0.3760244048790683,-1.2524466945486306,-0.18662037746961763
-0.4070782363503251,0.9215048035328627,-0.15678121556316238,-0.025366923089817472,-0.3021501300206099
-0.28436383804009513,-0.15559770594485633,0.2810671581684926,0.89176074607214,2.1537022686849476
0.18532564941538202,-2.1677519326301615,-0.0661565064367583,-0.2478382660053131,-0.3209687705185337
0.6191711169753933,0.11882509815314793,-0.871969976654656,-0.9014196827031773,-1.0696471711161588
-0.33925848388401536,-0.3324282385746148,-0.06324771595815296,0.38864285839277424,0.5407023033403081
1.0638515327343585,2.6279245165926186,-1.2432422136852415,-0.5185268868861845,1.3123945361297724
-1.141938226142404,-2.0281070259907583,-0.3967183236266304,-1.29730278927091,-2.179329669605573
0.006339062362268442,-0.3875080714180615,0.6458968205332676,0.8556758954348511,0.9922481292365681
2.5976737265958323,3.0046895438424803,0.6213382362718687,1.875304503832688,2.8293024149419943
0.2230797426252751,0.5615182126591857,1.9849824599378612,-0.6044028340982613,0.19425640316641443
1.4332145066061728,1.2799203268195112,-0.49726234939019703,1.9307312063600448,1.5634060553474491
0.09152017817320818,0.1603445261104823,0.265543773136406,1.0132369243554558,1.0178452178326203
0.5807770953297967,0.6203320863221629,0.15027487923069338,1.7023506801486157,1.3907290447278489
-0.056783194392691534,0.7492179330821902,0.12446771248124873,1.1321011008652062,0.9997528499987155
-0.17040758116420443,-0.8946901094748686,-0.38272920084720824,0.7489223159673795,-0.7138998127413081
-0.7794823967254196,0.48831738054549,0.25360024417311866,0.643670741803481,2.259760301936978
0.4303013589575532,-1.1578523374067626,0.500603042241671,-0.16864248660730136,-1.713667355355579
-0.8515371891857679,-0.09740062176134584,1.3485160497463404,-0.47461561564572274,0.14162810017621075
0.6655852363134291,0.5115335814430417,0.15120982992875512,0.13678731838809238,0.003919707614134127
1.08528700444591,-0.07364958596915294,0.7932211963808873,-1.0748862653897913,1.1852793437698277
0.36653140723722993,0.16058042829296823,-0.653592817778892,-0.650401351775674,-1.1483929989651935
-0.28624873355569186,-0.765664519443399,0.5362560968932133,-0.7814061291579195,-1.5087985860609914
0.4539655793086961,-0.8447185372073963,-1.0154070849892092,0.24283583550992144,-1.2707594141105627
-0.3086730555464126,-0.5079512693868506,0.938274555272077,0.9573752625311837,0.5481770475227767
0.9355471254651493,-0.6362150726390868,2.0197763556015134,-0.41957439041190525,-1.9009453910941378
-1.8314060842151236,-0.5929055780143138,-0.2999555613756859,0.27688334668253856,-0.6365396104236538
-0.3356073681186462,-1.3178891782116866,-2.5879102442709145,0.969059823566035,-2.3267805001771107
-1.9908119951239978,-0.2834289745709475,-0.6331016256161073,-0.8154416609311479,-0.3090139636229572
-1.495060830227205,-1.9290094830424227,0.6531722893765765,-0.7560662887902622,-1.8427527446726049
1.3638622298139094,0.11563619932594837,0.5877676468230294,0.09433157758508473,-1.3830728554763365
0.895184981971686,-0.17701091722933604,0.19581032350622873,1.4807249570770025,1.7374664228978918
-0.7194802332847904,0.9654066122324436,1.2982903396245722,0.5465038452485581,-0.5517116696444384
-1.502503456040897,-0.4209617824038566,0.358164904527265,-1.88125248536762,-0.3541634438575231
-2.964528837841651,-1.6939982214652198,0.7095257210115704,-1.5473384319870507,-2.32170915071394
-0.5434955079326346,0.22697168710340604,0.9875151827780073,1.3783491477298502,2.4042317953120538
2.4204150122474024,-0.8969848412059869,1.2526050003751628,-0.06442146742990781,0.26884738460819885
0.4348842714636474,0.17430854609091626,0.4110886477335292,0.565686000423823,-0.6380505976166585
-0.5595722860494895,1.717728034780018,1.4205778119279602,0.7809828034507136,0.3322728669050594
0.46508020950030626,0.3648866111528257,-0.19766108973993768,-0.7116119739053908,0.43216252384802745
-1.5609583529944429,-0.7125660073278859,-0.8152306137573716,-0.07955983686449897,-0.6584410528063592
-0.29732336269763543,0.39946669829337833,1.8266076499069868,-1.6885670115079625,-0.8138926636286576
0.09947747301573849,0.9453439798222147,0.4998078702693524,-0.3176048466216591,1.5861084283723756
-0.08610065182104851,-2.040517018800384,0.1927254602222639,0.19514087238327094,-1.6713769465785129
0.7908061216900806,-1.2869234209459866,-0.2623370118724058,0.1316961608114687,-0.4863198578918045
0.34464522623605237,0.38261390420417274,-0.4717314796587124,0.04630761773770031,1.1690885546232426
0.668326018107997,-0.3807135528435112,-0.10323117327065204,0.5153196401943124,0.8764205320610009
-0.6883722822307594,0.9678391211045345,1.7263457540990745,-0.22473743480236036,0.21193817563041462
0.8978154084105481,0.13373331432005015,0.5953600685754391,0.21634443688792993,-0.11386700524866819
1.6289369476239914,-1.120062261901122,-2.047491838300796,1.1069730739070454,-1.7453600565179699
-0.9701495196514126,-1.7530886766697034,-0.7411710595770657,-1.5714138553070238,-2.7184962586535595
-0.8876956557145598,-3.1935238474878553,0.8242366384583555,-1.6682291578464907,-2.1125172760924396
1.3357843363202329,0.4337666308772039,-0.0836668725867949,0.2787071331576575,0.2183546930751945
-0.19134398669506028,-0.2840561206119135,-1.8856955279177547,-1.379489913269988,-0.7568459383747244
1.403821392066557,0.27744798625138073,1.9762832812979578,-0.783586800388372,0.7861842025655649
-0.4425357118921839,0.8576111830836468,0.9396758647378867,1.5905550337820584,0.03675999937057828
1.4550455762707113,1.9260426246520492,0.5077999285828143,1.4900940516107042,1.4718880358987265
0.13148581680545218,-0.10464551495338512,-0.4028113905680842,-1.2454384842274908,-0.48633103617922024
0.2582288233226952,-2.0055792558155083,0.48067815304708295,-2.361253061787222,-2.887576096375881
1.5647180216699044,1.3996405913272105,-0.12209122282802937,0.9415259874868774,2.6020604033344608
-0.36177047744417123,0.37447498978648985,1.2605989214519353,1.3503256015674396,1.7709870972901123
-0.9411220959249583,-0.16116947780553414,0.4938444423825958,-0.632331443180573,-0.10661713340429807
-0.44856420802835434,0.7571976452939386,0.4718412190755315,1.3809411052979788,2.863301154624545
0.4523339506431387,-0.9386611211394706,-0.13608353100624582,0.1393832137570599,-1.8985342333013477
-1.5657590721805144,1.2258758500948816,0.19279471831428638,0.4533141325668348,0.8318687102251765
0.6374709026511215,-0.6406923018163073,0.8534324296643736,-1.5714426451686305,-0.7397305045584079
-0.5387713176177432,0.479837268391983,-0.7213916515842392,1.1121047641380757,-0.11858403052889277
1.1478126607335635,-1.11344136404657,-1.9559575360441803,-0.04477327746333004,-0.27839277893722164
-2.3942603049004716,-2.127394839720287,-0.023778329023611433,-0.7013879079426256,-2.5209738676193343
-0.7865657751041687,1.1230857591732517,0.9681640558191232,-0.7945382220118875,0.8548310868549353
-1.686468151234102,-1.104133898617901,0.1490009829263964,-0.8003448538927577,0.10559539187357725
-0.8262294663639526,0.42680787195082165,0.5357707312032853,0.16770311656811732,0.5695888953877979
0.24766590110894313,0.3607883392737177,-0.21230387655543254,0.10704256477738996,-1.618882363039392
-0.1792266254497549,-0.19457146872505013,0.7291112637680102,0.12712636520760817,-1.0005160729247555
-0.25337756894801494,1.2371615404586822,0.3255903353830638,0.08733620322994684,1.2269588750574751
-0.1591848713800608,-0.31625126034193074,-0.5819889389177573,-0.7069179308085185,-0.49038595206285884
0.20338824061994343,-1.6790439872534988,-0.8156045084314495,1.1580031617648379,-2.9545230304368184
-1.0085360419431078,-1.307508112923259,1.6283872648988542,-0.1022886443705644,0.5379320052213287
0.7068496408990508,0.9046165491512196,-1.4635911173000669,0.10575499034203961,0.6418422760004618
0.6626659703854839,0.72684209155525,0.7178407317623738,0.9144127869798833,0.0015527752675216222
0.385037937656101,-1.3745392814073878,-2.0914722120459337,-0.7586887435580911,-0.07165841491607527
0.5565334427512632,-0.44607579803256947,-0.24205607773274407,-1.1477437145034242,-1.4744497691748957
0.2964180008086595,-0.4335123487979169,-0.5636811897935627,1.0620661400811198,-0.3741752219626132
2.0350733027310675,0.6520670572513088,-1.8894081795044853,-0.45486917602666477,0.4569515916263582
-0.0870941710525099,-1.4194606741278064,-0.03680061180457889,0.3403035149705814,1.2414598796636822
-0.30708321833457675,-0.781912717286767,-0.614611877378226,-0.5346899895124931,0.9652858245732563
-0.7535275803573779,-1.1053742754382163,-2.4308186251927086,-0.39359086232448676,-0.3897710699772449
-1.0322626705778368,0.2554019244730009,0.9161886681161524,0.018995454615908514,0.5995843483947745
-1.2444717876010754,0.18688541718970608,1.4910718031458468,-0.015602918039357094,0.8771401816599995
-0.8887973132309185,-0.5902091147042752,1.3130616078852895,-1.5916783382569517,-2.3479274018499625
-0.07068038165207131,-0.4273799866967672,0.23074244464244098,0.47942101064724696,0.6701688491272224
0.3342951284977145,0.3070718762879706,0.13817970146350667,0.3088518867970247,-0.4261115866010305
0.051142058552161786,-0.791004078179494,1.0908553974419486,-1.775284886062577,-0.9646664153244549
-0.765535277429713,-1.5295997208822973,1.2235036690754093,-1.256075688976654,-1.620305332536813
0.9001845640199633,-0.4901289269157353,0.9865300248009043,1.2579768090384398,0.8851322681755667
0.7394126723009475,-0.26510682782098294,-1.0210696505760959,1.9965897989228067,0.983844950557372
-0.159648307422017,-1.2432979535032795,-0.17164453283606196,0.20646369054797276,-3.591447402298953
-0.652916144664712,-0.16439851326948127,0.39019659031058423,-1.530017054937693,-0.6683160723984065
0.5484279208297995,1.0039296341986081,-1.5084380786576468,1.3583947481158005,1.8233151434564672
0.18797358748610446,-1.4931115070551895,0.021427207504217283,0.30572293578210485,-0.14776593729808868
-1.4481272594150476,-0.6989349184914905,2.063311350431942,-0.4009664319887767,-0.6525366214487132
-0.0679802559844049,-0.253925923258538,0.5637583888232299,0.5502381648245355,1.7903178127134047
0.26203581207438104,0.47252659157399346,1.6151710589139543,0.3078502477166516,0.7015615487669836
-0.8996947864877538,-0.462786161347465,0.044078881724465585,-0.536251015660749,-0.06184528831100067
0.189843392837443,-1.3586775835129092,-0.4808215990376943,-0.9987211916504103,-0.13420749353109107
-1.4548224852577891,-1.98980054148606,-0.21566877236484244,-3.0871687262002885,-2.1758916341465757
1.3361861000121709,0.6975166255910582,-0.6065716603071152,1.1408185790148484,0.021149848888749823
1.2479499850594318,1.6527559122752251,-0.4679484135518211,1.3151298867904724,-0.9543945142467736
-0.25251733430296475,-2.3404460381944854,0.56417074571102,-0.5671919218860714,-1.9309398820583596
0.36345433783907316,0.5274537487211165,1.530578467482853,-1.3236415859492963,0.36285160726702664
-2.409921965799875,-0.015338977612294302,-0.9957080088710387,0.11504611030438783,-0.30324276507175907
-1.1563476602653329,-0.7311997063388795,0.7486388926522006,1.2654331354751853,1.0682950868498824
-0.2937789201521298,0.4033130359503759,0.034033935179091854,0.14483488207660436,0.4228021349337631
-1.0721330214268592,-1.2391355916112317,-0.031932102992938916,-0.8725730416805035,0.23221175361631707
0.7143964826306588,0.2378320420384526,0.5315682520465992,0.5121404672809599,0.07976954753904514
1.997296530747994,-0.6222652529341546,0.8076708922440721,2.2210857043122596,1.2670250522567406
-1.176614719429302,-2.9326506681335247,-0.7970769985768685,-2.3902770450667177,-2.1555787920647616
-0.8374634040851927,0.5240340991826788,1.5673722250567397,-0.15189289316992927,1.6876724756375678
0.23544836830993032,-0.8672707435694766,0.04923161668908177,-1.085777532570272,-2.6831614444041523
1.6111161484996208,0.3500302292463799,0.7291933412434465,-0.06949645284524608,-0.3181413863525907
-1.2223743125399031,-1.9976294095401448,-1.43255141439929,-0.9288440825801101,-0.6147206406716236
0.24903612230694197,-1.2854516419304047,1.7730235181257126,0.5532487179083369,1.1797950177676142
1.8212988508131087,0.736811725311938,0.6908684407985477,2.503931257897735,1.8604430042469167
-1.6517591481792673,-1.5554567682252802,0.2463603948775152,-0.718840386092689,-1.4298527415804716
-1.281069206845832,-0.26256360542821755,-1.6869548928400067,-0.5285499486805205,0.3615615916200757
-0.42360660646083825,0.24073511574917278,-0.7504814883543491,2.612047228670806,2.2950938474416724
-0.520588412855411,0.06055543989690165,1.256050319887801,-0.8107188932049431,-0.734743405321328
0.8126012877536446,0.644901434805266,0.48036154873418446,-0.1654672638583744,0.570024397650992
0.24165971982083806,1.6985980648423051,-1.4042003153762346,-0.7421658396203066,1.6587433750229685
-1.7749620596421283,-1.95507479886595,0.451767489324034,-0.008780466059081782,0.9085313608119359
0.5154104029008109,0.20277033524748767,-0.4829517037784771,1.3005671880962277,1.4942737209071497
-0.5775388873556505,0.8439203768417636,1.664493098929241,2.249213400860608,1.590705785516779
1.274447216941239,1.901763417808656,0.4826933467388971,-0.34302248936007107,0.7252952881863842
-0.6275875364369197,-1.7355574175630326,-0.3159605392483798,0.6388636873724965,0.009049692098354822
-0.6366152831022357,-0.6306630832069877,0.9897556536295521,-0.5710654305732488,-0.7165370993695684
0.5411316108045314,1.6286869012695262,-0.40888690047380444,1.701691840644067,1.854829869979185
0.7629264823369366,0.8895423463675336,-0.7259841039236796,-0.9178854370716407,-1.0720478098829282
0.4480993639362036,0.4193127134478746,-0.07070460636374495,1.7843776384980572,2.398902372464125
-1.6855973173848082,-1.060483352127061,0.5090277939831038,-0.24516134353105723,-1.7930070749777376
0.5380344399282511,-0.1147837057436199,-1.3653123643365765,0.34964928776212517,-0.31467896944375995
-1.034308051674399,-0.2289349627468753,-1.7367150887131593,0.5905893048026835,0.2784204318797179
0.2352761115882635,-0.1347360691903516,0.7833001062866868,-0.9054669164992482,1.4708853870438388
-1.4237344353990602,-0.7544458387439743,-0.829312064190196,-1.142476338860968,-1.3711440736547984
0.44632214754306365,0.3017597798865692,-0.30169807895431233,-0.148504441616987,-0.5893881448256052
-0.8065989043073726,-1.0813709806471854,-2.1812216961343216,-2.0910115860144995,-2.1370082006646363
-1.2826346502605217,-0.10918795124882252,-0.7117356997266289,0.5756400585710074,1.8921232212724048
0.7138201364823995,3.2711547166566195,0.9737791183658352,2.519445718116767,0.08295322388067206
0.2416445216767513,-1.6561643784876996,0.4335285440765823,-1.3780222534712125,-2.519831328128043
-0.6139768013928971,0.01702412349353133,-0.7661229291865794,0.3240442107616781,2.9018228762871563
1.4511788490210642,-0.03460066410837348,1.6469959209999845,-0.17342626976626335,0.4868686065543185
-0.44065242156974244,0.3570763667343787,-0.5774331141594871,0.4543954051357566,-0.7002490634041523
0.03210767397093466,0.06261131031332826,-0.6643084084911762,-0.46003873319618227,-0.24063169361619854
0.2689134476786009,0.8963114978817639,1.8062510012623314,-0.8054767228838162,0.1128719535668355
-0.6196659341286136,0.8656736094444728,0.47116223519725897,1.118212102523496,1.8380681018895904
0.47113629339992935,0.427989264468146,-0.4236279496711328,0.3333915986169712,2.220192434365122
-0.5334523471647244,0.8653261148318159,0.10446470326895133,2.1873107519829946,1.9148701146866207
-0.4116383222162092,-2.550117054896782,0.5880384610730247,-1.2148360460440406,-2.422264853483372
1.362642639803341,2.2717736178176935,2.14773490497099,-0.03812278227661048,1.9039098598687456
-1.040586052512433,-0.14523087436387644,-0.8603624415794189,1.028745199252542,-0.25649553441852824
-2.41278033092949,-0.12854680971956323,-0.08099785299497718,-0.12825308842165706,-1.1617374450596394
1.6109369950118944,-0.41440724733557444,-0.034165346533099245,1.051224132572382,-1.4415004293380413
2.549327952607003,0.44389409469816576,-0.8530718821071478,2.3035710789507156,-0.19139580767596032
-0.40526926544933184,-0.056696338579891786,-0.07269395422396184,-0.7825168180677584,-1.3160517344480969
-1.9368380406201853,-0.9854569096475556,-0.29031421244256617,-1.0444246425418522,-2.192730543795575
-0.31048397619594376,0.1278265999120177,1.178557458910341,-0.4491758761638375,0.15256188924880748
-0.2862229498406692,0.9414036860096315,1.4898461341140903,-1.2980211742867847,-1.3777105699626118
-0.18992384102225704,-1.8601822358139777,-0.8872640691032012,-0.8871873829698714,-4.5173511114166685
-1.1133880419218483,0.41712505075638495,-0.0888156286338962,1.186976191329603,-0.24186115354647164
0.5795611424764774,0.2980624811961657,0.7537110464144853,0.37968922973235947,-0.13298379225715679
0.5245073752860862,0.058471826518495335,-0.5676469009714629,-1.0517910044574712,-1.1021090768414532
-1.4944056198193065,-2.2183180442945436,-0.07685526155096034,-2.8761172630183527,-1.0276929743472636
0.6991967316948878,0.8526470342243089,0.1678876228980779,2.182016367366846,0.9037891415064272
2.052684981945891,-1.3877653600580917,-1.3857281857560482,0.2475259774566103,0.3108037718984572
0.17196033243705985,0.7343741774572425,-0.7365244596424738,-0.15476754305910762,0.2708207175182058
-0.33732516206850605,-1.660768416507245,0.9488281710789286,-1.2292623143362387,-1.4873200910943407
-0.1420032144075213,-0.017543694586286622,-0.16168458751918457,-1.4945892789702662,-1.5914467756263133
0.6152567669685612,0.18845954615636018,1.696891957745219,1.3436776065156883,0.7072859247552571
-1.7306716055072182,0.3042023445402746,-0.4223755573528277,-0.7060983699515195,-0.9660441396030394
0.16439070297730224,-0.09645981752523905,0.2622672558739217,1.3256813604467326,0.7846350857153173
-0.3904639494628101,-0.8893705358770707,2.1258655622551204,-0.8464087971933802,-1.1041392991589136
1.8478250129560454,0.4173098105068197,-0.9321938015112333,2.3372421629602282,0.4326480535419238
-0.17417273380495787,1.1857964018521847,-1.5420755793441212,0.9871763555112962,2.078380718382336
1.6678876111195198,1.3702407747406535,1.6976623637094292,1.5621627833593075,1.8930422590714058
-1.1037406977820072,-0.9375003304141649,-1.108059204517323,0.23458254549153323,-0.7995216266035641
0.5872591668027922,0.05718316343688268,-0.003551415075168974,0.8230523646825952,0.585610811095049
0.3194002637912288,-1.4074769586898326,-1.0081612980585346,0.0823960324558532,-0.7463629846189533
-0.8690472478647824,-0.046049423080850194,0.25434795403106103,-0.5628542505135719,-1.1720427740604848
0.17739611463694044,-1.6711176503769956,-1.4518715022988409,1.5905807662729123,0.030227786969122233
1.2125188368316004,1.3580558842512565,0.2274766773920708,-0.045454315824332125,0.03528902940608769
-0.3237917042866007,-0.03327425965203451,0.4398330486609239,-0.1941061825358191,-1.0586240567140262
-1.6919626491697632,1.5409704749890407,-0.5460177437775497,0.23288881920779272,-0.7358376831681082
-0.01756282647744591,-1.1089169669630288,-1.0327591197347976,0.1601569425620357,-0.8329873657391272
-0.9024230947925126,-0.5538736384368892,-0.5135449251295985,-0.8190400446810908,-0.11948618197526661
-0.3423409386638824,1.6271172315911215,-0.9406802647489418,1.05102696957666,0.6444127472050312
-0.08158776968512098,1.150595755448854,-0.07738225616409022,0.2823958373038572,3.237204488534325
-1.7056522160155172,-1.5490381595876583,-0.34591490797339564,-2.6440804644829385,-1.210873658382566
-1.615658339320678,-0.5838617412868913,-0.5747074639744958,-0.5937484192592143,0.7155250523874254
0.48206683552552526,0.24303277417784985,-0.1984666606783696,-0.8190388405656803,0.5072399788798434
-0.5227186961696804,0.020596519309316996,-0.013722715113790257,-1.8802182205533202,0.9230541243165712
-2.564744341820567,-1.9810526068951133,-1.2500592445008478,-1.174959175714661,-2.3650078747774295
0.7848440366081351,0.30310161560333304,0.7819930505478772,0.6881785891337647,-2.0650972832900725
0.27236975527423984,-1.4618674976395938,0.9974178755098977,-0.1758216054709687,-1.2649701599081644
-0.713874824036484,-0.22181396015015992,-0.5309245802404965,0.37046759179678435,0.6325453913708499
-1.3168302110582415,-0.2083944321072887,0.42267990184552046,-2.4955318374701023,-1.200446163040717
0.8358078963143372,0.1904845355132846,-0.18247179478066156,-0.9975863858517222,-0.12807831420143512
0.3493506225604296,-1.440375579304573,-0.06662532875286714,1.5155014576105441,1.3584729418228578
2.3826022826840734,2.1798330041934992,2.219729432461304,0.09555923991332471,1.5197536965159115
0.42018859872493736,0.5746193373552709,0.6810453442246801,0.8305362768513427,1.5014907346943285
0.3877031412620148,-0.5321635270699694,-1.9330635969306715,-0.396879370729095,-1.404084190752291
-0.16692793010856513,-0.3640644118923465,0.34477630787925495,-0.18498666609779468,-1.3699192282111072
0.816775867200791,-1.0922814691814433,0.9051793025545762,-1.0061698902968494,0.5648882432853413
0.6250852012481537,-0.7128675271538276,-0.435456428144251,0.55101098976141,0.019973266450859076
1.251725008565195,0.08805115792545803,1.0220501534954698,0.34310205235734675,-0.40765257830473245
-0.5213229186372847,0.9677593529615088,-0.3997877352634933,-0.13762536099811684,-0.20632749756542218
-0.4354074729439623,-0.8572071273594568,-1.7427012977301108,-1.1751369529616595,-1.6652951617988156
-0.4791031709119801,0.40066207387276864,0.26136764099447074,0.2320345475691304,-0.39261071032489225
0.7908017211789892,-0.1429153422720852,-1.7993513234279397,-0.6727741549121394,-1.6909925511957722
1.498374440296732,0.9971347810002409,0.059369421406480494,1.2639734023286584,-0.3698663105560589
-0.45884049314930064,-0.37081574871845246,1.1755975380299672,2.1053869539987353,0.6722812249857223
-0.42477372779309125,0.6606697361961978,-1.7115619848874215,1.829318224380048,0.6520681099677235
0.3140772238697992,-0.37777272628365827,-1.3928286137989954,0.9400276863014383,1.388286765452185
-0.24576150001290142,-1.6954090000088917,-1.540654945164245,-0.5293683587157257,-0.8193120260953922
0.95205365313989,-0.9059222047936877,-0.3350965211490663,-1.9484861087492202,-1.161886476693471
-2.2517772906190467,-1.657766139252281,-0.8630466274218935,-2.488316713302432,-2.8161618338318073
-0.8267050469168946,-1.6624124941365,0.9004126334551374,0.18973381768201758,-0.8573677540072169
-0.7824163339614458,0.019003848698828374,0.9600494837013459,-1.2088893375739513,-0.23394164080612112
-2.320356012386145,-1.6259461710511502,0.007489094753252878,-2.047235189251001,-1.1203949404384002
-0.9636384443688041,1.8927873278420295,0.7134665126964936,0.1596796196028504,0.9323416472040744
-0.9151561158459262,-2.195292653060095,-0.2653580189332012,-2.16163855103773,-1.0993418227247036
-0.20110465451883033,0.10308085145038605,-0.8310580998782592,-1.079656170351479,-0.48029835054260384
1.1129658916602636,-0.5714700776755371,0.6857693302539511,-0.08232620240434788,0.46594897847773736
-0.24506065119070236,-0.6159316659780297,-1.29682375415261,-0.9520555655899369,-1.2738458243731337
-1.0308137202711538,-0.37398738679503274,0.723180777863721,-1.3444938061509983,-2.671828963041657
-0.056954541213528984,-2.3538486454810514,0.47998790847451556,-1.8358144283983884,-1.8264725489211973
1.0491736214413494,0.7614129275087022,0.4392884803406978,1.0538197752022147,1.632940414502084
-0.9759588161149049,1.4149280542167153,0.24826535730303873,-0.008595185103750319,1.278513766364052
-0.9105748839214486,-0.9087257403293172,0.12769218621087813,-0.8407731664441908,-0.4525627919493139
0.5585489592742234,0.5390823364426212,-2.6997936691678865,1.520102253501776,0.6810053290172114
-0.22153611573080054,0.914153635259999,0.576504309063461,0.041860149339127156,-0.3287393591460292
0.6474846907094743,0.15978884532774063,-0.06541734424148485,-0.0070093542649373175,0.6281180145751821
-0.013646791795296761,-0.5877835844291316,-0.22696318838573049,0.2766355219370753,0.2708733849522469
0.701663648089567,-1.9249045796769688,0.1290611389947368,0.08292287831476619,-1.4818899415249276
-1.0350781140406409,0.12345166237465288,-0.037733422852331944,-0.40066958420046894,0.9708152282791489
-0.01208465078826744,-0.11921073512303096,-1.1759700771880675,0.2644921017724425,-0.8102248312228948
-0.21069001436315682,-1.1180288585697402,-0.4768113283963001,-1.196846511790911,-1.740961063971366
-1.2158909241041307,-0.48134875177437175,-0.07684914145148704,-1.4026199928152159,0.6944408400468711
-1.5634779478686884,-1.3654193307405256,-0.2946918876411449,-0.1871252371617801,0.09230965378275979
0.6857490352746618,-0.1855516134781947,0.10625388324160964,0.5551953510874204,-0.17318743708640977
-0.3509523822179781,0.43081496499971916,1.1686135344184692,0.2886160028673765,0.6470961935770982
-1.0222802816473295,-1.8495225521475094,-1.6408913186784004,-0.43746049289200034,-0.9886992779959165
-0.09617893692333225,0.9992258176120923,0.47546820144847757,0.32194628149309534,-2.0850278154247843
1.1280188573493026,-0.8336767344437523,1.0398266782851153,0.49338151859303603,-1.0766215573987399
-2.280737845371923,-1.8030703310412253,-2.2755325962495627,-0.0152342057553706,-0.9599988855075795
-1.4966386899033264,1.174626849114914,0.5320347355145277,0.8256801907627098,0.6435420433795058
-0.9228864418952676,0.6659115406947884,-0.3856050397646069,-0.03415822428373037,0.5629792230359496
1.461178866720329,0.8281934258556046,-0.12225235671642357,2.181074231610463,2.6341161416809538
0.2825869829192965,-0.7992367240415683,-1.1445785204268926,0.2224577910768613,-0.6099252511362809
0.7673172404612268,-0.1456136864692798,-0.21323482164122803,0.8469728544447002,-0.7069752121351547
-1.1401609191262305,-0.49680514497741884,1.6922140709336786,-0.5971252570806928,-0.22211354251801052
-1.1195358752705262,-0.5222447052956647,-1.421103772765913,-0.013531182662067015,-0.5945112775466693
0.4478137167227104,0.13303592873539236,-1.1296992238582098,-0.32225365896770763,-0.2506723659380603
0.05827433093544655,-0.0016953486164610551,-1.3024431747484428,0.9786436331285642,0.19670042310297292
0.5487388188641266,0.059994299726949196,1.7454165765461858,2.6848304451484193,3.6142501724486698
-0.18767099092845213,-1.5358142311042389,-0.9230839748525586,-0.8608052695398027,-3.4390140113199656
0.278143726454627,-0.37743887893351935,-0.13887072402773,0.031236426593474503,-0.5614351293026686
0.15811908490510918,0.9936835964394011,-0.10467786217441938,0.01963947213491468,0.5029725086387659
0.7777673890849254,-0.9794506030086547,-1.238258059305688,-0.3988183141787812,-1.4702769508764968
0.8070082819444706,0.21394862943295678,-0.17096605778454838,1.9063246260321063,1.9599624598608145
-1.6198719943917699,-1.629689925966729,1.9693366279465412,-2.2046759120209956,-1.103925240274223
-2.2472685553408556,-1.0122838482723164,1.08838576649256,-1.7708517325760564,-1.439322408912727
1.0017453976515072,0.11223951775893543,-0.8192659624645917,-1.0782202048393565,-0.7536640988839047
1.1877250932619365,2.0712562116056374,1.268950966543377,0.1734180947699132,1.0103803189561282
-1.0206230195133374,-0.0037283555664149315,-0.8326412793363009,-1.8495498851666548,-1.0314257740143455
-1.8598354065456286,-2.2417208690500257,-2.4488838995612285,-0.33305749045952426,-1.3657622078805376
0.09903482100720092,0.7207454221172991,1.2357345442998193,2.8705589281264254,1.607653073605063
0.9308382263505294,-0.3092327430867983,1.0673230237175113,-0.4245850277235849,-0.11445957081279884
1.7975945224343146,0.32180088983962374,1.1425925179997865,-1.2272777239240509,0.3004736477856515
0.516297572101451,0.5719678061578473,-0.29827490737221857,1.3433310399977512,-0.2726107245207776
-0.37171669658569495,-0.6776057680229822,-0.3700574644088809,-1.035470782233282,-1.907202174361281
-0.8931306215848627,0.44629050553052396,0.8836527053172805,1.033193405226835,-1.0087760741048442
0.011451291670654416,-0.7687844156780682,0.1608607126446867,-0.618279569088913,0.17286540797410888
-0.2992647038852564,-0.7685789045339915,-0.6242111426572418,-0.9041844335305059,-0.5860583758417857
-1.0150679057288161,-1.163904846412517,-1.9682432740419198,-0.5543912112209541,-0.35388051288396494
2.0487556508632867,0.45873907707660444,0.25149691726499357,1.5055997035728161,-0.44732884803875816
1.785168402072131,1.9917757565838783,1.6765423662112768,0.7695317192572789,3.1656018608007166
1.1360486835554284,1.1968206353880173,-0.838542098417752,0.49052657611168043,0.23482269760449115
-0.9208503978091276,1.2495187551075073,1.0136150082082291,-0.17800222192025295,0.703964073704883
0.8550193242972788,2.695153866047894,1.1400878549480609,-0.16530541178215596,2.119308887811241
0.6396264094979018,-0.7109038271833888,0.536877115077317,0.19718257125453179,-1.2438403837605003
0.44254562420405297,0.29169193942931537,0.9594830815478018,-1.7291827471215284,-1.2650697623587734
1.2496652670437771,-0.5513676831215444,-0.8598755468618162,-0.11779383812863675,-0.15540898602421632
0.6353711428612774,-0.4599203563176284,-1.7549355070602346,1.117015510574122,3.186020652592953
0.7400142363043355,-0.26007539210350167,-0.6115421433139298,0.5433308256454028,0.4427524596229748
0.6369062833722101,-0.7387017324521693,0.6675167977913565,0.3524084054589143,1.4132144871340908
0.3407914080021059,-0.8814915688973607,-0.8064513741951793,-0.15411460303808575,0.6878700592242324
-1.7836111297141835,-1.0360899433773465,-1.0048703068708458,0.42120092609449233,0.06056552785309166
0.08362107741055723,-1.1749078796811285,-2.173822705828977,0.9145869092863032,-1.1193562568280915
-0.5561919588491252,0.5640309797130703,1.9218568484648602,0.12052443878519309,1.4020313200673669
-1.2798409576400205,-1.5576740162375244,0.3584114365540526,-0.15115840378428358,-2.352526551658951
1.681816619300824,3.650477279746804,1.8650391651988056,2.342151619052785,4.597189573076887
1.7289953408857035,0.4131132194784566,1.0916593526230107,0.026148506185601272,0.4196361710502842
1.359220600295612,0.7516279420648582,0.8021967441628591,-0.7959209036092785,-2.6723700280703864
0.2552134168558746,-1.1677344760380979,-0.5946491979909585,0.1487622000823387,-0.8499922330097185
1.3506251209526567,2.381404318603968,0.5506571914589887,1.7937070562670152,2.741503508358432
0.012053180601825543,1.1255272711038289,-0.08990618326097616,1.5622800904128398,1.1068798077185675
0.20279727689777938,0.3983145091850022,2.754235107170501,-1.267003309736059,0.28066141071804496
-1.093471370461983,-1.893538234010557,-0.012149438903724552,-1.989457876239236,-2.791522222837192
0.39699130449884,0.4091625299740403,-0.34105168235922245,0.810873006118883,-0.6959646047411457
0.0603859284244171,-1.0912380818050464,0.13812534962476591,-0.5171715129727332,-0.8756502004364952
-1.3026521168344645,0.8110292050561285,-0.7675601563394789,-1.7986335323311202,-0.46223649634758107
-0.051197111816596905,0.7835961759448743,1.739585571372689,0.32527038631262356,0.2060489300664275
-0.07972956385680847,0.05557425655933188,-0.6443968188393405,-1.7252327177543962,0.3924619160446603
1.79756116664464,-1.5375950799250022,0.03980690864496511,-1.0289876155789783,-2.133498722891048
0.8942133285134095,-0.5363093144023259,0.9050848354224368,1.2244046785687122,0.6572132894155426
0.011445439091466015,0.6934838511456054,0.5928136995486744,1.9871434500830623,1.408283621777959
0.248787309801083,1.3372773466236973,0.49565319580674744,-0.7052398760435074,-1.2406972168459884
0.044212378104198057,1.1312760005442306,0.3501848850192843,0.06634559923240058,-0.7202193837801967
-0.20291398479554773,-1.6934699946392293,-0.5644820952788758,-0.582743251159875,-1.1261956066509209
-1.0824271973882869,-0.23966129188123936,-0.648596876491874,1.490353626910961,2.6065461662018246
-0.15105187760692926,0.9399892776794649,-0.16748664646586475,-0.12869147074152137,0.9849187153677827
-0.7460982566525134,-2.031384693908502,0.8701749654783845,-1.6873023706881098,-3.584978357442939
-1.2503155389231464,-0.6330622848348095,-0.29845531813559956,0.9800970464791141,-2.936970684297185
0.5112218145475347,0.5294533055822236,-0.626960277708106,0.11575076942051304,1.055655630505748
0.39126465993323495,-0.10513225488011743,-0.5478372733255438,0.9070956596594898,0.30476821946148713
-1.786707511368078,-2.7455156110852403,0.276715997371651,-3.166704662431277,-1.8148441672478892
-0.12268463749512722,-0.3870404288212923,0.5594945045180908,-0.47475320777299945,-1.3116147390085873
0.9957013858999235,0.17054526269493775,0.6114927570407382,-1.4149668688866002,-0.271998305237305
1.0592237388963208,0.38153946490040624,-0.1430593594396696,-0.5564390014387117,0.7587236947587099
1.0258368428852387,-0.05055300398245022,0.55819656897821,1.0150918926292127,0.03480580611689943
0.0389132069014598,-0.4235555497907374,-0.7706277898878692,-0.053528090954995805,-0.664627390603932
-0.8450453279691511,-0.4622289463315534,-0.39828271524524056,-0.6778543744272932,-0.5447788292810329
-1.0836988273580606,-0.09939480870931916,0.9399860447789496,-1.3016144046278533,-0.15782826010825746
0.3446049047503575,-0.13212431048148846,0.8346403857746179,-0.33705240316369445,-0.24892236133450238
0.379280034365163,-0.22000891118542037,-1.0479125193064294,0.5364761167973602,1.1322299405654563
1.2873311116050654,0.7360195940569908,-0.6347205746914786,1.7904674925043735,0.8013083540136521
1.0999796503420836,-1.3709261007241398,-0.30257567740383223,-2.783008093803067,-3.405362302726713
-0.1322286565311721,0.410920798960872,1.8494086143577715,-1.0769431745093667,1.6439641803389327
-1.2442058788608694,-0.6226660365533935,-2.102364975435012,-0.04275894100361005,-0.9373462253209481
-0.31910659701089783,-0.11985977190788709,0.24237500799813813,-0.29101400542833766,-0.7738789723858597
0.21723119481052466,1.0693499049997264,-1.0552383600732302,1.1073332349844192,1.0225382489198884
-0.2020832453894041,-2.05385715274295,-1.2504881393484306,-0.3645299207177613,-1.1420124270763745
-0.5778941080776558,-2.2627809238977914,-1.0432626952335788,-1.1746959259291048,-2.1054220674017308
0.2528848149698989,0.400246252808469,-0.3004344675943502,-0.23080235685394973,0.06728805102825058
-0.5039577414106104,-0.259095725833821,-0.6115246574093548,-0.37802217101034147,-0.3583211798502552
-0.628066083725374,-0.14385609867075477,-0.03594976922485628,0.9099700390817999,0.5578762972263295
0.31145283430343956,-0.3064006580207258,-0.6271757640367961,-0.6356971930156137,-2.828536384505486
-0.4019925801292048,0.4633573841753288,-1.323450169601554,0.26662849040772546,2.2458692442993766
0.24410438897835213,1.054691037480296,1.0962769237668653,0.459194560930672,0.6096544856709265
0.2732040228521227,0.6205843053775884,0.939047458762157,1.7027363671286988,1.7422829557001287
-1.1394290010915746,-0.8305429589503704,0.5940114223375579,-1.0931564266114453,-2.3399510343149323
-0.4812411238654122,-2.348309949547109,-2.2452274811282145,0.3241558352581366,-1.4830773183937365
1.4377769399164624,1.4881246779953594,0.7453555559827587,2.1094912452415526,1.1264920952980415
-1.1620795127342871,0.6406889098504492,0.8681855287620581,-0.3297396606231663,1.6251983891221866
-2.116654448970531,-0.4137688125012855,-1.1553327121899708,-0.7801562963304418,-1.1827761612517464
-1.8618451532425115,-1.8105650039887622,-0.5339962530144995,-1.5050339690135748,-0.8997194995610772
0.029109946576669203,0.6883493669560924,0.09023816477646197,0.010201963557562976,-0.30114478741636747
0.03091727803269891,0.8759883469777949,2.104947162661602,0.79292095024544,0.3818357089576734
-0.11761085542512094,1.5000108200865252,2.3744992830385305,0.32083119205488164,0.6648708408700175
1.2141898289285016,-0.26901500976370685,-1.7450608962593128,0.8409528331947997,1.5698140385448467
-2.672834433102938,-1.5628718010956075,2.025065465399307,-2.9572356538288207,-4.149906386492333
0.3959338744308236,0.945238988358867,-0.6316540246456639,-0.3442835147733079,-1.0151858184816531
1.5614382018035733,0.8381291723399444,1.232112046029409,2.700707545048148,-0.17788680699577064
-1.127780764891714,-1.6812179844971318,0.46613258551459136,-1.281873665657975,-2.569047944839004
-0.3798065117898951,0.22471302913175545,-0.9282763181022802,-0.6131734738421705,-1.0058302735473148
-0.7528919172286209,0.2022156752008562,0.5539324963993559,1.1753313612211822,-0.9257077358793698
-0.8943460460775288,-0.3592101267077624,-0.2152967822783927,-0.08711995936713296,-0.9573127896521166
-0.32626200502304,0.056453924668424166,0.5712661820329179,0.3886046488183957,-1.792185194190005
1.4274852836845047,0.4550367401565222,0.36718332442758234,0.3987765410009102,-0.591082083188965
1.8373866754875203,2.3969722734598196,1.9154311674081927,0.8082455879717413,2.8030293306259813
-0.3359392250401477,-1.0686617962327505,-0.24519899572376289,-0.6487930593576823,-0.45513935751411355
1.9050602140644701,0.3652093442684562,-0.058311266145774365,-0.6534741151427483,0.043335286735381645
0.035576476984624864,0.028773752019838054,1.1046963586631318,-0.28668423724920117,-1.131733798893796
1.753691026701081,1.755799541051433,1.1071358271740332,1.3030330360387083,-0.006668670499653251
-0.09329864965349177,-1.5440224949212826,0.2686066458445031,1.1511433118102925,-1.3355956670897757
0.1310578260461443,-1.2447678367423671,-1.0380170232813068,-0.16498731722162674,-0.7480874038780119
0.36547447264283456,-1.8549912714133006,-1.6017601628224643,-2.646179176418282,-1.3824084441891231
3.1788536793675353,1.6221220649370385,-0.08108552377159,1.5247366017547725,2.475796750481226
0.8512855748447891,-0.6926630297841494,-0.8286202946158334,2.8835486525712493,1.7260551738768828
-0.7072645299514636,-0.7381823749936485,0.5647445561548198,0.24596501477117727,-0.9836655441791797
0.9689978702758579,1.145877125029637,1.3197471878029428,-0.32578817027456514,0.43459430978152397
-0.3618325370843415,-0.8224589893250174,-0.4018679863299056,0.23394675484122845,-0.6155986472791404
-0.48975018290120914,-0.013593068707240002,1.4268410982121804,-1.179743926885435,-0.7978582612910878
0.9086013598921112,0.15496277219433358,1.4149582254234299,-1.3022336477861767,1.4634924591638914
0.031085587771152785,0.6876571089820133,0.4496960490951432,-0.15392005312894969,-1.4951317090770808
0.2785759145118061,0.6144160246039119,-0.6375669641618665,0.5483790162663411,0.34672070451286563
0.013985221865292462,-0.6849410852419637,1.369894323862951,-2.4159152347431836,-0.33867925279641686
0.3365831266054347,-0.00036019009773527455,-0.061125408057728814,-1.0361033469845538,-1.9841232635746828
0.4249658974361492,1.256226346438569,0.5030331003625453,0.539709250916118,0.25422037341109316
-1.9369669327054153,-0.5751065960986106,-0.5121370044361518,-1.1168784933527927,-1.3750310422933412
0.6665726686095336,-1.0636757234463647,-1.208893043331107,0.4110913910166263,-0.5495818882688537
-0.9820194198303006,0.16460835125623235,-1.4192569249384221,-1.1595794277674742,-2.093442697861838
-1.4422991355594112,-1.8057601902159437,0.09114026649250495,-0.45910812149940106,-0.537148095519633
-0.05841124385868244,1.3833074480316647,-0.368408819487121,-0.5742777271077096,1.0708021429751216
0.08398100331445427,1.066089341498252,3.379358031446299,1.4184927332420072,1.7506663623880772
-0.6935061645722808,-0.056867606367739576,-1.1031338645228008,-1.7259304033925364,-1.7617726791712194
0.8310318576679744,0.5727487722701806,-1.085557843300316,-0.3970336355148848,0.17436835916494303
-1.342175166370907,-0.8801198392128735,0.24610276384957913,-0.3865502353345081,-1.5327752768427783
-0.40694374877255896,0.47505207701374863,1.67182910207142,1.3978051372570859,0.6038319463639046
-0.5848741349330007,-0.663215236928562,-0.13437491413534697,-0.09883031630808486,1.7724451505525545
-0.04658727033933937,0.2122762354702242,0.6359252908283416,0.5066071751030107,0.4551370828828734
0.2788642846472388,1.0113665088430706,-1.5401374648057886,0.8137702835656161,-0.27495817978283
-1.0078929496235447,-1.065753018337362,-0.9972374511929754,-1.5556200560986064,-1.1303511965540705
0.7242826774562282,-0.25405140126258047,-1.1498311495163411,-0.1445078756042748,-1.5510755644967373
0.06300236833469973,-1.1628642965571092,-2.243905138915014,-1.2093737003207077,-2.38512939095392
-1.8919462533119105,-2.0104458727821273,-0.4861535278107512,-2.5532414873211455,-3.3229668160004273
-1.958591440397031,-2.807691031640018,-3.0851349311024387,-0.6918779392348823,-2.1940939376331334
-0.012319292929311235,0.723669142550874,-1.1050952233221865,-0.6163870682819681,0.9906100327236544
-0.2209673177863541,1.6984308831895938,-0.027664529056539933,0.9381690704746006,1.8966981748489362
-0.1034201135746335,1.3736188866829888,-0.41920016994617965,-0.6995611976310563,0.8600241727732946
-0.02798245653021798,-1.6400757168916253,-0.09561421286041988,-0.3489191610268769,-2.0934220045497938
0.22555983951018072,0.8376345913112904,0.5568705692958671,-0.1973348944445656,-0.2211645070022863
0.9476496098482997,1.8959311548477062,1.4625780923897687,2.156767260066263,2.298350314340173
-1.1111049009175582,-1.012362431197443,0.4486750033534452,-0.5042925478458687,-1.3866474749100688
-1.1719715929007948,-0.4845383658003479,0.9954512796212764,0.49128581287950857,-0.2749927167615181
-1.0933238578873825,-2.5545549797483615,-0.7682848939335626,-1.9518088869040395,-1.811823240796251
0.2889166671687069,0.41119450156099496,0.4268252512849682,0.2219310907621837,-0.03220988503971267
1.2449473786464456,-0.44108604011149466,0.3791878898857429,-0.8598749385440911,-0.4244209002257676
-0.4313394382395105,-0.016034701208240715,0.1368462870285535,-0.5767902121052783,1.296315999362809
-2.5016998441596545,-2.812052844268708,-1.2408101631976862,-1.8892732810604713,-2.522521861651587
-1.7039928626103755,-0.30913820682759185,-0.42518399638804777,-1.2756745629236792,0.3869314248435707
-0.8330367227960116,0.607994314028684,-0.6351601041213603,-1.0951841292487527,0.3157751596973238
-0.5577288874952148,-1.062044305462775,0.10300353827099712,-2.0262544197405266,-3.0113913560884176
-0.4083886272547264,-1.3411168950377976,-0.4963502425964948,-2.899441413379635,-3.22241618347224
0.038581122980917065,-2.1367324810595227,-1.3041934949546217,-1.6386264200420675,-2.7212753336206914
-0.3117463308224471,-0.42228669948640063,-0.7889308291705128,0.5228414834646526,-0.8045575249436574
1.0493396105434072,-0.056924129009901514,-1.227475966485759,-1.0609711071570695,-0.6492373641902998
-0.6760173056813688,-1.889028065188739,0.412132921680578,-2.408965848369327,-1.534352908697211
-0.862317959475543,0.021859776574400136,0.0851756025041076,1.5239119735191906,2.028044291751518
0.4788755385204616,-0.21053261260441836,-0.9259598033545956,-0.3918544726629638,0.0069815460913911775
-1.5356420699261193,-1.843556993608292,0.2554806515195307,-0.9281655395602231,-1.8315408226374674
0.3896917053200022,-0.359726552980446,-0.6683667756885068,0.7490073920420183,-1.887600824099088
0.10255318372626686,1.1225811307927491,-1.069726287774408,1.3562697947711415,1.3100150381992834
-0.1474424899181674,0.5292156987961385,1.1799869988579745,-1.5908580978366493,-0.5050510058252641
1.5882761071492624,0.6641119920761414,-0.33786339194327064,0.5786406274643991,0.8043050633092279
-0.6222074452947179,0.3080012958174936,-0.3164987014635414,-1.6183892444871018,1.4510463233685287
2.0602981705240784,-0.13076287112469975,0.00195115901176756,3.0117119298153137,2.6810710755383793
-0.2254314492280993,-0.08034562388701912,1.118485535624009,-1.9183400554636394,-0.2142819214253434
-1.2770164449702281,-0.4911858919986435,-0.12731847673328134,-1.6019558138136898,-2.0086479829777666
0.0699195809948253,-0.4348767180013142,-1.3295801415456245,-0.9276596394344578,-1.1942629279146295
-1.0762452003187681,-0.10266426420196623,0.5130777248515903,-1.9632152874696154,-1.1277843827642928
-0.7517558478135002,-1.1563113785512014,-0.8318831378270845,0.8309192315574316,-0.9381031975072385
0.39703330001549114,0.7083373577838634,2.1004251788646253,2.4048077984287635,2.0570371030914316
0.5555820886501248,0.8296498158563543,0.24777522275901612,-0.026769610926841436,0.758348951680461
-0.6221677002542582,-3.2282437308241954,-3.203508617145925,-2.1576692258211807,-3.7812642713773803
0.9874051450464006,1.684008276033318,0.8625674549656693,-1.4396141357919077,1.3190137792168295
1.157507758021134,-2.468878686342187,1.2039341863575723,-1.125194702924151,-1.6947263608243088
1.436301756563291,0.10210185272552019,1.6065508443630456,-0.16594432397485764,-1.76099238236692
0.5294134071257222,1.2900547378206162,2.21797071010275,-0.4200443511147315,0.3836623350079729
1.3634287120856987,0.3644276307904087,-0.910431896144341,0.9582612991469697,0.942550920891594
-1.8807984279740577,-2.4065851964565708,0.1901768370398076,-1.1697358315673207,-0.8153681525762053
-0.31790654862846257,-1.2526903360309019,-1.0978016405459823,-0.0538045129538024,-0.9907238276189684
-0.8670052563234263,0.5934031181589066,0.40231033486459644,-0.4193272685748718,0.9149761922392246
0.11922586112739209,-1.8618915999849421,-1.8315199087352152,-1.4443067294829781,-1.4659571779637444
-0.5714492191967373,0.8432664913636486,-0.3914758174736217,0.9100000920878785,-0.4384213173118159
-0.16615784194973238,-0.13818102001715554,-0.717701003368762,0.05093419950350091,-1.458790183739341
1.8821748802040494,-0.2513078903404088,0.8231629130404936,1.1878542501516027,1.5908330500549457
-0.1697197964203783,-0.6679749705128802,0.6003266305157914,0.3519042962378075,1.0379488412442326
0.4137923171902139,-0.8103281825065343,0.03372127457886051,-1.6284999018084372,-0.8965048117527628
-0.2322693299242352,0.750090535896411,-1.1253942572864624,-0.9787879406190169,1.3813580188187256
0.07571328795683367,0.7702844537412551,-0.08055409782734346,-0.40860213129150763,1.2374510102664804
0.006016680436071459,-1.716152793658011,-1.2609623210401293,-1.4887290574150251,-0.9188121960567971
0.4483240659731595,-0.17943008553275433,0.8158467310346648,0.6548076153480138,-0.6736227238260422
1.165307535547177,-0.2805318945706886,1.094025284445978,0.36988412193459974,1.776104864714299
1.6473939976033294,2.117206967261035,1.6915476762091934,2.101975444728885,2.6458926382032333
0.30962008190916684,-1.1952890771718376,-1.113806142568572,0.23401843962864763,-0.8664740933832167
0.5895468860356021,1.899002547702815,-2.1698323579786347,1.5790811383609027,1.0707718377265607
-1.150864509185567,-0.6087269941097211,-1.9495799963629707,1.0008956292172062,0.45286809906759945
-0.08787674241562057,-1.205872777587898,1.0598651448058392,-0.25095782298596475,-2.3994743851253864
0.940289464883589,-0.20712945156748336,-0.46339377479006894,0.360014777233816,0.776115938186029
0.8659686384171532,-0.368634521559874,-0.8613149747330747,1.7650528117784967,-0.2553616695847155
0.21160973279505021,-1.168813535617981,1.2920162327508398,-0.5472648578060552,-0.48743108788773126
0.8863939595679524,0.5380238395143518,1.4265670973426325,-0.10852630071032165,-0.3068715679561444
0.49076670899381886,-0.9825321961949339,-0.2703109369386935,-1.0295439371697388,-1.27045677940704
1.2003062582593216,-1.5802130796085279,-0.7674670149465322,0.584525507669103,-1.2547536587347772
0.2893591569571111,-0.23499689236961752,-0.20260028052401047,1.2703795880865574,0.3564619166776684
-0.35569831557103465,-1.9807053542033999,-0.36704880787355393,-0.8761692762113897,0.4609422308099591
0.33584125693302364,-0.5842402396010086,1.4247752094393056,0.8900168631314811,1.5379479388138768
-2.9305943758527757,-2.374400076529744,-0.28985259872169644,-3.740982855594967,-2.7032125110668987
0.38288573638504947,-0.6797732038332894,0.5220970718891546,-0.31691634980354666,-0.5236390423833704
-3.6484128252147836,0.17675668898276187,1.222297115765,-1.3083755072287666,0.27839178156988265
-1.723463407405418,-0.8053201365331383,-1.4112720419843603,-1.2839070408453557,0.08535680541830959
0.45176860094098414,0.4712553094658242,0.8962384853137901,2.52864604498774,0.6701322950192529
0.47752933726679525,-1.5376336277088969,-0.7558588429742804,-0.3868155139974253,-1.0396634384853634
-1.1624287801399305,-1.2351775237068872,-2.886648452884832,-0.6732007245522009,-2.7761473536976244
-0.7121020418789388,-0.1407483909979574,-2.1703940592851994,0.1901109563276803,1.1477833929257564
1.370540988614132,-0.3392584186059404,-1.1198951371004677,-1.5033757819638724,1.1437071574159492
-0.48403013157267616,0.6660477777316474,0.7310375619120756,-0.7848653661064741,0.020673830435672155
2.242920312402985,-0.0812364296995376,0.08942863643043517,1.6057357640003977,-1.3627604577525863
-0.001919853176346149,-0.2873260624533487,0.0981688884396594,-0.7640855135549499,0.08318472857288173
0.4080361768812668,-0.38203355153492524,1.8363334172262145,-1.2265444137951045,-1.708378148814095
1.6168743984986076,0.9727578586312198,-0.9772622572704464,2.37914355097056,1.01260126313643
0.13102711619341956,-0.5749765137587162,-0.19992630831133842,1.1895571919686359,0.9338667288959637
-1.0023439806908012,-0.772065617416338,0.07281583758891422,-0.10710213653041656,-0.18502975146065287
-0.10972745169429012,-0.2909938176251306,-0.8235712618258747,-0.4397188442318316,-0.8534651062670828
-0.03561073619009093,-0.7521931254863465,-0.19733119256296297,-0.5115652976552968,-1.281484844172486
-1.3647416522015654,-0.19727695760367003,-1.659132421365446,-0.9606567560038075,-0.453130801582906
-0.25583207134082336,0.1104723391670136,1.443047919025132,-1.3159433885691127,-1.0122010772134171
-0.7421920939974367,-0.45970999225188564,-0.8195275502059995,-1.2870526033672176,-1.982618431412559
0.9243577571874341,0.031720545131758926,2.0409163283180307,-0.7362421272398799,-0.6848239725324206
0.03461187418525562,0.8138499676126248,0.2116806663663324,-0.7411335746573848,0.37595081973280414
-0.28279574914757005,-0.3772321183879664,0.9148839298827932,-1.9824134766191652,-1.357926264323199
-0.10618193062257626,-0.35978825313722534,-0.16123934493232198,0.07588065829870713,-1.5554623956674443
0.22312201840616613,0.21625103909120907,0.0341314794668329,0.38120630615840395,1.287793294000346
0.6168141996897798,2.0049937828157405,0.01746880162271436,2.6843407711370038,-0.3941441599669617
-0.9997122286693814,-1.8402123223558946,-0.4329397673404907,-1.6177490124297769,-2.1753458224212707
-1.0415876326212719,0.24906307674471873,0.39271761743746303,0.004706007047283911,-0.4933139126579146
1.10467921540253,-1.34083155353684,-1.186507974976232,0.345085317476071,-0.34969424015896966
-0.4123368722047306,0.011104722126258654,-0.12171299922200661,-0.3487541340920475,0.39283710181984105
-1.416842092596874,0.8666617813102937,-0.3501003316640256,0.7359657398275365,1.9280785688299504
0.44381270938310463,0.7105748791588594,2.5128365333624965,0.6900356446566902,-0.012099611444385427
0.4633698060326869,0.4023099815561363,0.03096222455522235,0.621153584718756,1.098314120470928
-1.5307152572567064,-1.3258070974314289,1.1474900224671307,-1.4746751256908053,-1.2044227236796559
0.2294817154636063,0.1457880582203574,-0.1518859133415811,-0.19999318795899396,-1.7386419400525415
0.7355830924283736,0.11572873081083052,0.2940062607781571,-1.1595887308998463,1.378610255389602
0.37438645170626306,-0.5987833820782238,-1.5009990862687363,-0.632915378683355,-2.054624507620989
0.6319814837613953,0.026115126405827804,-0.4392380539461163,-0.3341194174217149,0.2654951268017714
-1.404269098090749,-0.503753388418935,0.05940865426356626,-1.758635578076946,-0.7412345390501929
0.3310401549961956,-0.6016969910325425,1.405494733650104,0.017458270043485102,-0.016482578165451567
-0.30261975947044656,-0.9575917004199074,-0.06529473244970102,0.7358642074678532,-0.6054406286376346
-0.4827901587246448,-1.6016385169544969,0.90322880610665,-1.8742006479496722,-0.9106669313115823
0.8705556094197265,0.8260028307987594,0.888529959442548,0.6973032590015492,-0.7987144290429412
1.47927451067423,1.2849605173041754,2.0128894946559384,-0.4789224783369588,0.33719023080968535
1.794370054600359,1.128845471536697,0.27961142415595364,0.6290047492244337,0.3150504338281068
1.314807871505905,1.6225744284816184,2.8769950391962897,-0.22225645417261491,0.022861045087470222
-0.10973418163018368,0.03300054626563568,-0.47549603903443227,0.6772748204305271,0.7838854708016272
0.35272016302634307,0.5683710188789283,-0.5959505285339906,-0.22643361167590592,1.2484472463211942
0.7668228717842437,0.6872488615296847,-1.0038781912301287,1.7173824580998194,0.8590598979778858
0.12117794929946196,-0.03095104476847689,2.321739358127527,-0.6860956784052566,0.4221577283389107
0.13076418791154437,-0.4654397341738751,0.679815652365735,-0.9325134773640645,-0.6492359403243714
0.823753131876182,2.626029181730002,0.4536880211386921,-0.046505254318045974,0.3970297544679695"""

df = pd.read_csv(io.StringIO(CSV_TEXT))

EDGES = [
    ('Z', 'X'), ('Z', 'Y'), ('X', 'M'), ('X', 'Y'),
    ('M', 'Y'), ('X', 'W'), ('Y', 'W'),
]
COEFFS = {
    'Z->X': 0.5, 'Z->Y': 0.3, 'X->M': 0.3, 'X->Y': 0.4,   # X->Y is true direct effect
    'M->Y': -0.2, 'X->W': 0.5, 'Y->W': 0.5,
}
data = {
    'df': df,
    'edges': EDGES,
    'coeffs': COEFFS,
    'n_samples': len(df),
    'seed': 42,
    'nodes': ['Z', 'X', 'M', 'Y', 'W'],
}

print(f'5-node DAG with {len(data["edges"])} edges')
print(f'Sample size: {data["n_samples"]}')
print(f'Nodes: {data["nodes"]}')
print(f'Edges: {data["edges"]}')
print(f'True direct effect X -> Y: {data["coeffs"]["X->Y"]}')
print(f'True total effect X -> Y: {data["coeffs"]["X->Y"] + data["coeffs"]["X->M"] * data["coeffs"]["M->Y"]:.3f}')
print()
print('Head of dataset:')
print(df.head())


5-node DAG with 7 edges
Sample size: 1000
Nodes: ['Z', 'X', 'M', 'Y', 'W']
Edges: [('Z', 'X'), ('Z', 'Y'), ('X', 'M'), ('X', 'Y'), ('M', 'Y'), ('X', 'W'), ('Y', 'W')]
True direct effect X -> Y: 0.4
True total effect X -> Y: 0.340

Head of dataset:
          Z         X         M         Y         W
0  0.304717  0.093076 -0.424028  1.462474  1.030979
1 -1.039984 -1.249279 -1.040661  0.084118  0.312638
2  0.750451 -0.039247  0.422236  2.091117  1.299256
3  0.940565  1.104193  0.583112 -0.940609  2.320624
4 -1.951035 -0.972524 -1.696549 -0.900371  0.493340


In [2]:
from IPython.display import Markdown, display
display(Markdown(
    '## Step 1: Marginal correlations\n\n'
    '先看 X, Y, Z, M, W 两两之间的边际相关系数。'
    'X 与 Y 显著正相关（同时受 Z 与 M 影响）；'
    'W 与 X、W 与 Y 也正相关（W 是 collider，X 与 Y 共同决定 W）。'
))

corr = df.corr().round(3)
print('Marginal correlation matrix:')
print(corr)

## Step 1: Marginal correlations

先看 X, Y, Z, M, W 两两之间的边际相关系数。X 与 Y 显著正相关（同时受 Z 与 M 影响）；W 与 X、W 与 Y 也正相关（W 是 collider，X 与 Y 共同决定 W）。

Marginal correlation matrix:
       Z      X      M      Y      W
Z  1.000  0.432  0.165  0.406  0.339
X  0.432  1.000  0.307  0.436  0.585
M  0.165  0.307  1.000  0.005  0.151
Y  0.406  0.436  0.005  1.000  0.587
W  0.339  0.585  0.151  0.587  1.000


In [3]:
from itertools import combinations

def partial_corr(df, x, y, given):
    """Pearson partial correlation of x and y given a list/set `given`."""
    if not given:
        return df[[x, y]].corr().iloc[0, 1]
    from numpy.linalg import lstsq
    Z = df[list(given)].values
    Z = np.column_stack([np.ones(len(Z)), Z])
    res_x = df[x].values - Z @ lstsq(Z, df[x].values, rcond=None)[0]
    res_y = df[y].values - Z @ lstsq(Z, df[y].values, rcond=None)[0]
    return float(np.corrcoef(res_x, res_y)[0, 1])

nodes = ['Z', 'X', 'M', 'Y', 'W']
print('Pairwise marginal correlations:')
for a, b in combinations(nodes, 2):
    print(f'  {a} ~ {b}: {df[a].corr(df[b]):+.3f}')

Pairwise marginal correlations:
  Z ~ X: +0.432
  Z ~ M: +0.165
  Z ~ Y: +0.406
  Z ~ W: +0.339
  X ~ M: +0.307
  X ~ Y: +0.436
  X ~ W: +0.585
  M ~ Y: +0.005
  M ~ W: +0.151
  Y ~ W: +0.587


In [4]:
display(Markdown(
    '## Step 2: d-separation 三种基本结构\n\n'
    'd-separation 定理刻画的是**路径**是否被条件集合阻断。先在孤立的 3 节点结构上演示定理，'
    '再回到 5-node DAG 看多路径叠加效果。'
))

# Isolated 3-node structure demos: each built with FRESH data so no extra edges.
rng = np.random.default_rng(2024)

# (1) Pure chain: Z -> X -> Y, no other edges
Z_chain = rng.normal(0, 1, 1000)
X_chain = 0.5 * Z_chain + rng.normal(0, 1, 1000)
Y_chain = 0.5 * X_chain + rng.normal(0, 1, 1000)
chain_df = pd.DataFrame({'Z': Z_chain, 'X': X_chain, 'Y': Y_chain})

# (2) Pure fork: Z -> X, Z -> Y (Z is confounder)
Z_fork = rng.normal(0, 1, 1000)
X_fork = 0.5 * Z_fork + rng.normal(0, 1, 1000)
Y_fork = 0.5 * Z_fork + rng.normal(0, 1, 1000)
fork_df = pd.DataFrame({'Z': Z_fork, 'X': X_fork, 'Y': Y_fork})

# (3) Pure collider: X -> W <- Y, no other edges
X_col = rng.normal(0, 1, 1000)
Y_col = rng.normal(0, 1, 1000)
W_col = 0.5 * X_col + 0.5 * Y_col + rng.normal(0, 1, 1000)
col_df = pd.DataFrame({'X': X_col, 'Y': Y_col, 'W': W_col})

print('=== Pure chain Z -> X -> Y (test: Z _||_ Y | X) ===')
print(f'  corr(Z, Y)            = {chain_df["Z"].corr(chain_df["Y"]):+.3f}')
print(f'  partial_corr(Z, Y|X)  = {partial_corr(chain_df, "Z", "Y", ["X"]):+.3f}')
print('  -> controlling the middle X blocks the Z->X->Y path: Z and Y become independent')
print()

print('=== Pure fork Z -> {X, Y} (test: X _||_ Y | Z) ===')
print(f'  corr(X, Y)            = {fork_df["X"].corr(fork_df["Y"]):+.3f}')
print(f'  partial_corr(X, Y|Z)  = {partial_corr(fork_df, "X", "Y", ["Z"]):+.3f}')
print('  -> controlling the common cause Z removes confounding: X and Y become independent')
print()

print('=== Pure collider X -> W <- Y (test: X _||_ Y | W opens path) ===')
print(f'  corr(X, Y)            = {col_df["X"].corr(col_df["Y"]):+.3f}  (marginally independent)')
print(f'  partial_corr(X, Y|W)  = {partial_corr(col_df, "X", "Y", ["W"]):+.3f}  (CONTROL OPENS PATH)')
print('  -> conditioning on collider W induces spurious X-Y correlation')
print()

print('=== 5-node DAG (multi-path) ===')
print(f'  corr(X, Y)            = {partial_corr(df, "X", "Y", []):+.3f}')
print(f'  partial_corr(X, Y|Z)  = {partial_corr(df, "X", "Y", ["Z"]):+.3f}')
print(f'  partial_corr(X, Y|W)  = {partial_corr(df, "X", "Y", ["W"]):+.3f}')
print('  -> controlling Z removes Z\'s confounding but X->Y (direct) and X->M->Y remain')
print('  -> controlling W (collider) still induces spurious X-Y correlation')

## Step 2: d-separation 三种基本结构

d-separation 定理刻画的是**路径**是否被条件集合阻断。先在孤立的 3 节点结构上演示定理，再回到 5-node DAG 看多路径叠加效果。

=== Pure chain Z -> X -> Y (test: Z _||_ Y | X) ===
  corr(Z, Y)            = +0.228
  partial_corr(Z, Y|X)  = -0.010
  -> controlling the middle X blocks the Z->X->Y path: Z and Y become independent

=== Pure fork Z -> {X, Y} (test: X _||_ Y | Z) ===


  corr(X, Y)            = +0.234
  partial_corr(X, Y|Z)  = +0.024
  -> controlling the common cause Z removes confounding: X and Y become independent

=== Pure collider X -> W <- Y (test: X _||_ Y | W opens path) ===
  corr(X, Y)            = -0.003  (marginally independent)
  partial_corr(X, Y|W)  = -0.214  (CONTROL OPENS PATH)
  -> conditioning on collider W induces spurious X-Y correlation

=== 5-node DAG (multi-path) ===
  corr(X, Y)            = +0.436
  partial_corr(X, Y|Z)  = +0.316
  partial_corr(X, Y|W)  = +0.142
  -> controlling Z removes Z's confounding but X->Y (direct) and X->M->Y remain
  -> controlling W (collider) still induces spurious X-Y correlation


In [5]:
def ols_coef(df, y_col, x_cols):
    """Plain-OLS coefficient on the FIRST regressor (treatment X = x_cols[0]).

    Other regressors (Z, M, W) are included in the regression but their
    coefficients are discarded. This is the coefficient on the **cause** of
    interest, not the last covariate in the formula.
    """
    X = df[x_cols].values
    X = np.column_stack([np.ones(len(X)), X])
    y = df[y_col].values
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return float(beta[1])  # intercept=beta[0]; treatment X=beta[1]

direct = data['coeffs']['X->Y']
total = direct + data['coeffs']['X->M'] * data['coeffs']['M->Y']
print(f'True direct effect X -> Y = {direct:.2f}')
print(f'True total effect  X -> Y = {total:.3f}  (= direct + X->M * M->Y)')
print()

# Naive regression (no adjustment): confounded by Z + the indirect X->M->Y path
naive_coef = ols_coef(df, 'Y', ['X'])
print(f'Naive Y ~ X:          b_X = {naive_coef:+.3f}  (true total = {total:.2f})')
print('  -> b_X inflated by Z (confounder) and the X->M->Y indirect path')
print()

# Backdoor adjustment: control {Z} only (Pearl 1995 criterion)
bd_coef = ols_coef(df, 'Y', ['X', 'Z'])
print(f'Backdoor Y ~ X + Z:   b_X = {bd_coef:+.3f}  (true total = {total:.2f})')
print('  -> recovers the TOTAL causal effect of X on Y (direct + indirect via M)')
print()

# Wrong: also control M (mediator) -> over-adjustment closes X->M->Y path
wrong_coef = ols_coef(df, 'Y', ['X', 'Z', 'M'])
print(f'Wrong Y ~ X + Z + M:  b_X = {wrong_coef:+.3f}  (true direct = {direct:.2f})')
print('  -> over-adjustment closes X->M->Y, recovers only DIRECT effect (not total)')
print()

# Wrong: control W (collider) -> induces spurious correlation, biases b_X toward 0
collider_coef = ols_coef(df, 'Y', ['X', 'Z', 'W'])
print(f'Wrong Y ~ X + Z + W:  b_X = {collider_coef:+.3f}  (true total = {total:.2f})')
print('  -> collider W opens spurious X-Y path, biases b_X toward 0')

True direct effect X -> Y = 0.40
True total effect  X -> Y = 0.340  (= direct + X->M * M->Y)

Naive Y ~ X:          b_X = +0.442  (true total = 0.34)
  -> b_X inflated by Z (confounder) and the X->M->Y indirect path

Backdoor Y ~ X + Z:   b_X = +0.325  (true total = 0.34)
  -> recovers the TOTAL causal effect of X on Y (direct + indirect via M)

Wrong Y ~ X + Z + M:  b_X = +0.370  (true direct = 0.40)
  -> over-adjustment closes X->M->Y, recovers only DIRECT effect (not total)

Wrong Y ~ X + Z + W:  b_X = +0.065  (true total = 0.34)
  -> collider W opens spurious X-Y path, biases b_X toward 0


In [6]:
coefs = [naive_coef, bd_coef, wrong_coef, collider_coef]

## 实战小结

**纯结构 d-separation（3 节点演示）**：
- Chain Z → X → Y：marginal corr(Z, Y) = +0.23，partial_corr(Z, Y|X) = -0.01（链被关）
- Fork Z → {X, Y}：marginal corr(X, Y) = +0.23，partial_corr(X, Y|Z) = +0.02（叉被关）
- Collider X → W ← Y：marginal corr(X, Y) ≈ 0（天然独立），控制 W 后 = -0.21（对撞打开伪路径）

**5-node DAG 多路径演示**：
- Marginal corr(X, Y) = +0.44：Z (confounder) + X→Y 直接 + X→M→Y 间接叠加
- partial_corr(X, Y | Z) = +0.32：控制 Z 后只剩 X→Y 直接 + X→M→Y 间接（≈ 0.34 总因果效应）
- partial_corr(X, Y | W) = +0.14：控制 collider W 引入伪路径

**Backdoor 调整（5-node DAG，全部是 b_X）**：
- Naive Y ~ X：b_X ≈ +0.44，被 Z（confounder）与 X→M→Y 间接路径双重抬升
- **Backdoor Y ~ X + Z：b_X ≈ +0.32，恢复到 X 对 Y 的总因果效应（≈ 0.34）**
- 错误调整 M（mediator）：b_X ≈ +0.37，只恢复直接效应（≈ 0.40），丢失 X→M→Y 间接路径
- 错误调整 W（collider）：b_X ≈ +0.07，被 X→W←Y 伪路径拖向 0

局限：
- 真实业务里 DAG 节点画错任何一个，整个识别都失效（领域知识决定）
- 高维 DAG（20+ 节点）需要 Shpitser et al. 2010 的 ID 算法自动找调整集
- 反馈循环系统（动态 DAG）需要 do-calculus + 时间切片，本文未涉及